# TDK 2026: Integrált Ökonometriai és Gépi Tanulási Számítási Rendszer
**Kutatási tárgy:** Vasúti és közlekedési infrastruktúra hedonikus ármodellje és térbeli hatásmechanizmusai (Budapest X. Kőbánya és Bécs II. Leopoldstadt)  
**Funkció:** Teljesen moduláris, konfigurációs regiszterekkel vezérelt, univerzális számítási platform.

---

### A moduláris felépítés és izoláció alapelvei:
1. **Főrészekre bontott architektúra:**
   - **1. FŐRÉSZ:** Leíró statisztikák, áreloszlások, normalitás ($K^2$), sávos bontások, korreláció és VIF
   - **2. FŐRÉSZ:** 11 hedonikus OLS specifikáció becslése, modellválasztás (AIC/BIC), White-teszt, RESET, robusztus standard hibák
   - **3. FŐRÉSZ:** Térbeli vetületi koordináták (HD72 / MGI), $W$ súlymátrixok, globális Moran's I, Anselin LM diagnosztika, SAR és SEM Maximum Likelihood becslés, LeSage & Pace közvetlen/közvetett hatásfelbontás
   - **4. FŐRÉSZ:** Lokális paraméterbecslés (GWR & MGWR), adaptív sávszélesség ($BW$), lokális $R^2$, 5-számos paramétereloszlások
   - **5. FŐRÉSZ:** Gépi tanulási benchmark (ElasticNet, Random Forest, XGBoost, LightGBM, CatBoost), 5-Fold CV, SHAP TreeExplainer globális és infrastrukturális hatáscsoportok
2. **Különálló univerzális becslő motorok és specifikációs szótárak:** Minden egyes számítás és modell önálló egységként áttekinthető, paraméterezhető és ember által módosítható.
3. **Objektív módszertan:** Zéró empirikus narratíva; kizárólag matematikai formulák (LaTeX), hipotézisrendszerek ($H_0 / H_1$) és tiszta táblázatos riportok.
4. **Kettős kimenet (Dual Output):** Minden eredmény megjelenik a képernyőn és strukturáltan exportálódik CSV, JSON és Markdown formátumban a `tables/` könyvtárba.


In [ ]:
# ==============================================================================
# INTEGRÁLT RENDSZER-INICIALIZÁCIÓ ÉS CSOMAGOK TELEPÍTÉSE
# ==============================================================================
!pip install -q gspread statsmodels matplotlib seaborn openpyxl scipy tabulate geopandas libpysal spreg mgwr shap xgboost lightgbm catboost scikit-learn

import os
import sys
if hasattr(sys.stdout, "reconfigure"):
    try:
        sys.stdout.reconfigure(encoding="utf-8")
    except Exception:
        pass
if hasattr(sys.stderr, "reconfigure"):
    try:
        sys.stderr.reconfigure(encoding="utf-8")
    except Exception:
        pass
import json
import warnings
import datetime
import urllib.parse
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats

# Ökonometriai és statisztikai könyvtárak
import statsmodels.api as sm
from statsmodels.stats.outliers_influence import variance_inflation_factor
import statsmodels.stats.diagnostic as smd

# Térinformatikai és térbeli ökonometriai könyvtárak
import geopandas as gpd
from shapely.geometry import Point
import libpysal
from libpysal import weights
import esda
import spreg
from mgwr.gwr import GWR, MGWR
from mgwr.sel_bw import Sel_BW

# Gépi tanulási és XAI könyvtárak
from sklearn.model_selection import train_test_split, KFold, cross_val_score
from sklearn.linear_model import LinearRegression, RidgeCV, ElasticNetCV
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
import xgboost as xgb
import lightgbm as lgb
import shap

warnings.filterwarnings('ignore')

# Kettős környezet detektálása (Google Colab vs. Lokális Windows)
IS_COLAB = ('google.colab' in sys.modules or os.path.exists('/content'))
BASE_DIR = '/content' if IS_COLAB else os.path.abspath('.')
TABLES_DIR = os.path.join(BASE_DIR, 'tables')
FIGURES_DIR = os.path.join(BASE_DIR, 'figures')
os.makedirs(TABLES_DIR, exist_ok=True)
os.makedirs(FIGURES_DIR, exist_ok=True)

def log_info(message: str) -> None:
    """Időbélyeges technikai állapotüzenet kiírása."""
    now_str = datetime.datetime.now().strftime('%Y-%m-%d %H:%M:%S')
    print(f"[{now_str}] [INFO] {message}")

def export_table(df: pd.DataFrame, table_id: str, title: str = None) -> pd.DataFrame:
    """Többcsatornás export: strukturált CSV, JSON, formázott MD és publikációs LaTeX (.tex) formátumba."""
    if title:
        try:
            print(f"\n=== {title.upper()} ===")
        except Exception:
            print(f"\n=== {title.encode('ascii', errors='replace').decode('ascii').upper()} ===")
    try:
        from IPython.display import display
        display(df)
    except Exception:
        try:
            print(df.to_string())
        except Exception:
            print(df.to_string().encode('ascii', errors='replace').decode('ascii'))

    csv_path = os.path.join(TABLES_DIR, f"{table_id}.csv")
    json_path = os.path.join(TABLES_DIR, f"{table_id}.json")
    md_path = os.path.join(TABLES_DIR, f"{table_id}.md")
    latex_dir = os.path.join(TABLES_DIR, 'latex')
    os.makedirs(latex_dir, exist_ok=True)
    tex_path = os.path.join(latex_dir, f"{table_id}.tex")

    df.to_csv(csv_path, index=False, encoding='utf-8')
    df.to_json(json_path, orient='records', force_ascii=False, indent=2)
    try:
        df.to_markdown(md_path, index=False)
    except Exception:
        with open(md_path, 'w', encoding='utf-8') as f:
            f.write(df.to_string())

    # Automatikus LaTeX booktabs export
    try:
        cols = df.columns.tolist()
        col_align = "l" + "r" * (len(cols) - 1)
        tex_lines = [
            f"% Auto-generated publication table for TDK 2026: {table_id}",
            "\\begin{table}[htbp]",
            "  \\centering",
            f"  \\caption{{{title or table_id.replace('_', ' ').capitalize()}}}",
            f"  \\label{{tab:{table_id}}}",
            "  \\small",
            f"  \\begin{{tabular}}{{{col_align}}}",
            "    \\toprule",
            "    " + " & ".join([f"\\textbf{{{c}}}" for c in cols]) + " \\\\",
            "    \\midrule"
        ]
        for _, r in df.iterrows():
            row_vals = [str(r[c]).replace('&', '\\&').replace('%', '\\%').replace('_', '\\_') for c in cols]
            tex_lines.append("    " + " & ".join(row_vals) + " \\\\")
        tex_lines.extend([
            "    \\bottomrule",
            "  \\end{tabular}",
            "  \\begin{tablenotes}",
            "    \\footnotesize",
            "    \\item \\textit{Megjegyzés:} Robusztus standard hibák zárójelben (HC3). *** p<0,01, ** p<0,05, * p<0,10.",
            "  \\end{tablenotes}",
            "\\end{table}\n"
        ])
        with open(tex_path, 'w', encoding='utf-8') as tf:
            tf.write('\n'.join(tex_lines))
    except Exception:
        pass

    log_info(f"Táblázat sikeresen elmentve: {table_id} (CSV, JSON, MD, LaTeX)")
    return df

log_info(f"Számítási platform inicializálva. Környezet: {'Google Colab' if IS_COLAB else 'Lokális Windows'}")


In [ ]:
# ==============================================================================
# INTEGRÁLT ÉLŐ ADATBETÖLTÉS ÉS EGYSÉGES TÍPUSKONVERZIÓK (MODELL FÜL)
# ==============================================================================
import urllib.parse
ID_KOBANYA = "1rKaXN3OBDrh49vv1sJujGLchqGIJXJNqF1vf_MFwdok"
ID_LEOPOLDSTADT = "1vCDtFCIysMN6zt2pZy7247ycTRhmjZABtL3_HwOuc7A"

def parse_hungarian_number(val):
    if pd.isna(val) or val is None or val == '':
        return np.nan
    if isinstance(val, (int, float)):
        return float(val)
    s = str(val).strip().replace('\xa0', '').replace(' ', '').replace(',', '.')
    try:
        return float(s)
    except ValueError:
        return np.nan

def load_live_modell_dataset(spreadsheet_id, city_name):
    df = None
    try:
        import gspread
        from google.auth import default
        creds, _ = default()
        gc = gspread.authorize(creds)
        ss = gc.open_by_key(spreadsheet_id)
        ws = ss.worksheet("Modell")
        rows = ws.get_all_values()
        headers = [h.strip() for h in rows[0]]
        df = pd.DataFrame(rows[1:], columns=headers)
    except Exception:
        encoded_sheet = urllib.parse.quote("Modell")
        csv_url = f"https://docs.google.com/spreadsheets/d/{spreadsheet_id}/gviz/tq?tqx=out:csv&sheet={encoded_sheet}"
        df = pd.read_csv(csv_url)

    df.columns = [str(c).strip().lower().replace(' ', '_') for c in df.columns]

    numeric_cols = [
        'wgs84_lat', 'wgs84_long', 'ar', 'alapterulet_nm', 'fajlagos_ar', 'ln_fajlagos_ar',
        'szobak_szama', 'kulteri_alapterulet_nm', 'epites_eve',
        'dist_fout_m', 'ln_dist_fout_m', 'dist_vagany_m', 'ln_dist_vagany_m',
        'dist_vasutallomas_hm', 'ln_dist_vasutallomas_hm'
    ]
    for col in numeric_cols:
        if col in df.columns:
            df[col] = df[col].apply(parse_hungarian_number)

    # Dinamikus sávos dummy és log-távolság újraszámítás a folytonos távolságokból (konzisztencia-garancia)
    if 'dist_fout_m' in df.columns:
        df['ln_dist_fout_m'] = np.log(np.maximum(1.0, df['dist_fout_m']))
        df['d_fout_0_150'] = (df['dist_fout_m'] <= 150.0).astype(int)
        # Főútvonal lecsengő externália (150 m után 0)
        df['road_prox_censored'] = np.maximum(0.0, np.log(150.0 / np.maximum(1.0, df['dist_fout_m'])))

    if 'dist_vagany_m' in df.columns:
        df['ln_dist_vagany_m'] = np.log(np.maximum(1.0, df['dist_vagany_m']))
        # Andersson et al. (2010) & Day et al. (2007) szerinti cenzorált közelségi mutató (300 m után 0)
        df['track_prox_censored'] = np.maximum(0.0, np.log(300.0 / np.maximum(1.0, df['dist_vagany_m'])))
        df['d_vagany_0_150'] = (df['dist_vagany_m'] <= 150.0).astype(int)
        df['d_vagany_150_300'] = ((df['dist_vagany_m'] > 150.0) & (df['dist_vagany_m'] <= 300.0)).astype(int)
        df['d_vagany_300_500'] = ((df['dist_vagany_m'] > 300.0) & (df['dist_vagany_m'] <= 500.0)).astype(int)

    if 'dist_vasutallomas_hm' in df.columns:
        df['ln_dist_vasutallomas_hm'] = np.log(np.maximum(1.0, df['dist_vasutallomas_hm']))
        df['d_vasutallomas_0_375'] = (df['dist_vasutallomas_hm'] <= 375.0).astype(int)
        df['d_vasutallomas_375_750'] = ((df['dist_vasutallomas_hm'] > 375.0) & (df['dist_vasutallomas_hm'] <= 750.0)).astype(int)
        df['d_vasutallomas_0_750'] = (df['dist_vasutallomas_hm'] <= 750.0).astype(int)
        df['d_vasutallomas_750_1125'] = ((df['dist_vasutallomas_hm'] > 750.0) & (df['dist_vasutallomas_hm'] <= 1125.0)).astype(int)
        # Carlos Moreno (2020) 15-perces cenzorált állomáselérhetőség max(0, ln(1125/dist)) - TOD elérhetőségi prémium modellezése
        df['station_access_censored_1125'] = np.maximum(0.0, np.log(1125.0 / np.maximum(1.0, df['dist_vasutallomas_hm'])))

    # Ingatlanjellemző dummyk típuskonverziója
    dummy_cols = [c for c in df.columns if c.startswith('d_') and not c.startswith('d_vagany_') and not c.startswith('d_vasutallomas_') and c != 'd_fout_0_150']
    for col in dummy_cols:
        def clean_dummy(x):
            if pd.isna(x) or x is None or x == '':
                return 0
            val_str = str(x).strip().replace(',', '.').split('.')[0]
            if val_str in ['1', '10']:
                return 1
            try:
                return 1 if float(val_str) >= 1 else 0
            except ValueError:
                return 0
        df[col] = df[col].apply(clean_dummy)

    log_info(f"{city_name} [Modell fül] betöltve. Összes sor: {len(df)}, Geokódolt: {df['wgs84_lat'].notnull().sum()}")
    return df

# Alapadatbázisok betöltése a Modell fülről
df_kobanya = load_live_modell_dataset(ID_KOBANYA, "Budapest X. Kőbánya")
df_leopoldstadt = load_live_modell_dataset(ID_LEOPOLDSTADT, "Bécs II. Leopoldstadt")

# Geokódolt pontos részhalmazok szűrése (érvényes koordináta, pozitív alapterület és meglévő fajlagos ár)
df_k_pontos = df_kobanya[df_kobanya['wgs84_lat'].notnull() & (df_kobanya['alapterulet_nm'] > 0) & df_kobanya['fajlagos_ar'].notnull()].copy()
df_l_pontos = df_leopoldstadt[df_leopoldstadt['wgs84_lat'].notnull() & (df_leopoldstadt['alapterulet_nm'] > 0) & df_leopoldstadt['fajlagos_ar'].notnull()].copy()


# ==============================================================================
# 1. FŐRÉSZ: LEÍRÓ STATISZTIKA ÉS ADATDIAGNOSZTIKA
# ==============================================================================


## 3. Modul: Térbeli Dummy Esetszámok és Mintasűrűségi Diagnosztika

### Módszertani elv:
A távolsági indikátorváltozók (dummy változók) ökonometriai becslésénél alapvető követelmény a kellő mintanagyság ($N_d \ge 5$). Amennyiben egy távolsági kategóriában kevés megfigyelés található, a kovarianciamátrix szingulárissá válhat, a standard hibák torzulnak.

A vizsgált térbeli zónák:
1. **Vasúti zajzónák (síntől mért távolság):**
   - $D_{\text{sín}, 0-150\text{m}} = 1$, ha $\text{dist\_vagany} \le 150\text{m}$
   - $D_{\text{sín}, 150-300\text{m}} = 1$, ha $150\text{m} < \text{dist\_vagany} \le 300\text{m}$
   - $D_{\text{sín}, 300-500\text{m}} = 1$, ha $300\text{m} < \text{dist\_vagany} \le 500\text{m}$
   - Referencia kategória: $> 500\text{m}$
2. **Vasútállomás elérhetőségi zónák (gyalogos hálózati távolság):**
   - $D_{\text{állomás}, 0-375\text{m}} = 1$ (5 perces izokrón)
   - $D_{\text{állomás}, 375-750\text{m}} = 1$ (10 perces izokrón)
   - $D_{\text{állomás}, 750-1125\text{m}} = 1$ (15 perces izokrón)
   - Referencia kategória: $> 1125\text{m}$


In [ ]:
# ==============================================================================
# MODUL 3: TÉRBELI DUMMY ESETSZÁMOK SZÁMÍTÁSA ÉS MINTASŰRŰSÉG-ELLENŐRZÉS
# ==============================================================================
SPATIAL_DUMMY_DEFS = [
    ('d_vagany_0_150', 'Vasút sín [0 - 150 m]'),
    ('d_vagany_150_300', 'Vasút sín [150 - 300 m]'),
    ('d_vagany_300_500', 'Vasút sín [300 - 500 m]'),
    ('d_vasutallomas_0_375', 'Vasútállomás [0 - 375 m]'),
    ('d_vasutallomas_375_750', 'Vasútállomás [375 - 750 m]'),
    ('d_vasutallomas_750_1125', 'Vasútállomás [750 - 1125 m]'),
    ('d_fout_0_150', 'Főút közelség [0 - 150 m]')
]

def compute_spatial_dummy_counts(df: pd.DataFrame, city_name: str, sample_name: str) -> pd.DataFrame:
    rows = []
    total_valid = len(df.dropna(subset=['dist_vagany_m'])) if 'dist_vagany_m' in df.columns else len(df)
    for col, label in SPATIAL_DUMMY_DEFS:
        if col in df.columns:
            cnt = int((df[col] == 1).sum())
            pct = (cnt / total_valid) * 100 if total_valid > 0 else 0.0
            rows.append({
                'Város': city_name,
                'Minta': sample_name,
                'Változó': col,
                'Leírás': label,
                'Esetszám (N)': cnt,
                'Relatív arány (%)': round(pct, 2),
                'Min. N>=5 Kritérium': 'Megfelel' if cnt >= 5 else 'Alacsony (összevonandó)'
            })
    return pd.DataFrame(rows)

df_dummy_counts_k = compute_spatial_dummy_counts(df_k_pontos, "Budapest X. Kőbánya", "Pontos")
df_dummy_counts_l = compute_spatial_dummy_counts(df_l_pontos, "Bécs II. Leopoldstadt", "Pontos")
df_dummy_summary = pd.concat([df_dummy_counts_k, df_dummy_counts_l], ignore_index=True)

export_table(df_dummy_summary, "1_terbeli_dummy_esetszamok", "Térbeli dummy változók megoszlása és esetszámai")

## 4. Modul: Leíró Statisztikai Mutatók Számítása

### Módszertani specifikáció:
A folytonos változók helyzet- és szóródási mutatóinak definíciói ($N$ elemű minta esetén):
- **Mintabeli átlag:** $\bar{x} = \frac{1}{N} \sum_{i=1}^N x_i$
- **Korrektált tapasztalati szórás:** $s = \sqrt{\frac{1}{N-1} \sum_{i=1}^N (x_i - \bar{x})^2}$
- **Medián és kvartilisek ($Q_1, Q_2, Q_3$):** Rendezett minta alapján számított percentilisek ($25\%, 50\%, 75\%$).
- **Relatív gyakoriság (dummy indikátorokra):** $p = \frac{1}{N} \sum_{i=1}^N d_i \times 100\%$


In [ ]:
# ==============================================================================
# MODUL 4: AKADÉMIAI LEÍRÓ STATISZTIKAI TÁBLÁZAT GENERÁLÁSA
# ==============================================================================
CONTINUOUS_VARS = [
    ('ar', 'Ingatlan ára (HUF M / EUR ezer)'),
    ('alapterulet_nm', 'Alapterület (m²)'),
    ('fajlagos_ar', 'Fajlagos ár (HUF/m² / EUR/m²)'),
    ('ln_fajlagos_ar', 'ln(Fajlagos ár)'),
    ('szobak_szama', 'Szobák száma (db)'),
    ('kulteri_alapterulet_nm', 'Kültéri méret (erkély/terasz m²)'),
    ('dist_fout_m', 'Főút távolság (m)'),
    ('ln_dist_fout_m', 'ln(Főút távolság)'),
    ('dist_vagany_m', 'Vasúti vágány távolság (m)'),
    ('ln_dist_vagany_m', 'ln(Vasúti vágány távolság)'),
    ('dist_vasutallomas_hm', 'Állomás távolság (m)'),
    ('ln_dist_vasutallomas_hm', 'ln(Állomás távolság)')
]

DUMMY_VARS = [
    ('d_felujitott', 'Felújított állapot dummy'),
    ('d_felujitando', 'Felújítandó állapot dummy'),
    ('d_lift', 'Lift jelenléte dummy'),
    ('d_kulteri_kapcsolat', 'Erkély / terasz kapcsolat dummy'),
    ('d_panel', 'Panel építési technológia dummy'),
    ('d_szuteren', 'Szuterén szint dummy'),
    ('d_foldszint', 'Földszinti elhelyezkedés dummy'),
    ('d_zaroszint', 'Zárószint / legfelső emelet dummy'),
    ('d_epult_2015_utan', 'Új építés (2015 után) dummy'),
    ('d_fout_0_150', 'Főút 150 m-en belül dummy'),
    ('d_vagany_0_150', 'Vasút vágány [0 - 150 m] dummy'),
    ('d_vagany_150_300', 'Vasút vágány [150 - 300 m] dummy'),
    ('d_vagany_300_500', 'Vasút vágány [300 - 500 m] dummy'),
    ('d_vasutallomas_0_375', 'Vasútállomás [0 - 375 m] dummy'),
    ('d_vasutallomas_375_750', 'Vasútállomás [375 - 750 m] dummy'),
    ('d_vasutallomas_750_1125', 'Vasútállomás [750 - 1125 m] dummy')
]

def calculate_descriptive_statistics(df: pd.DataFrame, city_name: str, sample_name: str, is_huf: bool = True) -> pd.DataFrame:
    rows = []
    # 1. Folytonos változók kibővített paraméterei (Q1, Q3, Ferdeség, Csúcsosság)
    for col, label in CONTINUOUS_VARS:
        if col in df.columns:
            s = pd.to_numeric(df[col], errors='coerce').dropna()
            if len(s) > 0:
                scale = 1e6 if (col == 'ar' and is_huf) else (1e3 if col == 'ar' else 1.0)
                s_scaled = s / scale
                q1_val = float(s_scaled.quantile(0.25))
                q3_val = float(s_scaled.quantile(0.75))
                skew_val = float(stats.skew(s_scaled)) if len(s_scaled) > 2 else np.nan
                kurt_val = float(stats.kurtosis(s_scaled)) if len(s_scaled) > 3 else np.nan
                
                rows.append({
                    'Város': city_name,
                    'Minta': sample_name,
                    'Változó': label,
                    'Típus': 'Folytonos',
                    'N': len(s_scaled),
                    'Átlag': round(float(s_scaled.mean()), 2),
                    'Szórás': round(float(s_scaled.std()), 2),
                    'Min': round(float(s_scaled.min()), 2),
                    'Q1 (25%)': round(q1_val, 2),
                    'Medián (Q2)': round(float(s_scaled.median()), 2),
                    'Q3 (75%)': round(q3_val, 2),
                    'Max': round(float(s_scaled.max()), 2),
                    'Ferdeség': round(skew_val, 3) if not np.isnan(skew_val) else np.nan,
                    'Csúcsosság': round(kurt_val, 3) if not np.isnan(kurt_val) else np.nan,
                    'Relatív arány (%)': '-'
                })

    # 2. Dummy változók
    for col, label in DUMMY_VARS:
        if col in df.columns:
            s = pd.to_numeric(df[col], errors='coerce').dropna()
            if len(s) > 0:
                cnt = int((s == 1).sum())
                pct = (cnt / len(s)) * 100
                rows.append({
                    'Város': city_name,
                    'Minta': sample_name,
                    'Változó': label,
                    'Típus': 'Dummy (0/1)',
                    'N': len(s),
                    'Átlag': cnt,
                    'Szórás': np.nan,
                    'Min': 0,
                    'Q1 (25%)': 0,
                    'Medián (Q2)': int(s.median()),
                    'Q3 (75%)': 1 if pct >= 25 else 0,
                    'Max': 1,
                    'Ferdeség': np.nan,
                    'Csúcsosság': np.nan,
                    'Relatív arány (%)': f"{pct:.1f}%"
                })
    return pd.DataFrame(rows)

df_desc_k_pontos = calculate_descriptive_statistics(df_k_pontos, "Budapest X. Kőbánya", "Pontos", is_huf=True)
df_desc_l_pontos = calculate_descriptive_statistics(df_l_pontos, "Bécs II. Leopoldstadt", "Pontos", is_huf=False)

export_table(df_desc_k_pontos, "2_leiro_statisztika_kobanya_pontos", "Leíró statisztikák – Budapest X. Kőbánya (Pontos minta)")
export_table(df_desc_l_pontos, "2_leiro_statisztika_leopoldstadt_pontos", "Leíró statisztikák – Bécs II. Leopoldstadt (Pontos minta)")


## 5. Modul: Áreloszlások és Transzformációk Normalitásvizsgálata

### Matematikai specifikáció és teszt-hipotézisek:
A lineáris OLS regresszió standard feltevése a hibatagok normális eloszlása. Az ingatlanárak eloszlása jobbra ferde (log-normális), így az árak logaritmizálása indokolt.

Alkalmazott statisztikai mutatók és tesztek:
- **Ferdeség (Skewness):** $\sqrt{\beta_1} = \frac{\frac{1}{N} \sum (x_i - \bar{x})^3}{\left(\frac{1}{N} \sum (x_i - \bar{x})^2\right)^{3/2}}$
- **Csúcsosság (Kurtosis):** $\beta_2 = \frac{\frac{1}{N} \sum (x_i - \bar{x})^4}{\left(\frac{1}{N} \sum (x_i - \bar{x})^2\right)^2} - 3$ (excess kurtosis, normális eloszlásra értéke $0$).
- **D'Agostino-Pearson $K^2$ teszt:** $K^2 = Z_1(\sqrt{\beta_1})^2 + Z_2(\beta_2)^2 \sim \chi^2(2)$
- **Nullhipotézis ($H_0$):** A vizsgált változó normális eloszlású ($p < 0.05$ esetén $H_0$ elvetve).


In [ ]:
# ==============================================================================
# MODUL 5: NORMALITÁSVIZSGÁLATOK ÉS ELOSZLÁSI TESZTEK (K² ÉS JARQUE-BERA)
# ==============================================================================
def compute_normality_metrics(series: pd.Series, variable_name: str, city_name: str) -> dict:
    s = pd.to_numeric(series, errors='coerce').dropna()
    skew_val = float(stats.skew(s))
    kurt_val = float(stats.kurtosis(s))
    k2_stat, k2_pval = stats.normaltest(s)
    jb_stat, jb_pval = stats.jarque_bera(s)

    return {
        'Város': city_name,
        'Változó': variable_name,
        'N': len(s),
        'Átlag': round(float(s.mean()), 2),
        'Szórás': round(float(s.std()), 2),
        'Ferdeség (Skewness)': round(skew_val, 4),
        'Csúcsosság (Excess Kurtosis)': round(kurt_val, 4),
        "D'Agostino K² Stat": round(float(k2_stat), 4),
        'K² p-érték': round(float(k2_pval), 6),
        'K² Döntés (α=0.05)': 'Normális (p >= 0.05)' if k2_pval >= 0.05 else 'Nem normális (p < 0.05)',
        'Jarque-Bera Stat': round(float(jb_stat), 4),
        'JB p-érték': round(float(jb_pval), 6),
        'JB Döntés (α=0.05)': 'Normális (p >= 0.05)' if jb_pval >= 0.05 else 'Nem normális (p < 0.05)'
    }

norm_rows = []
for city, df in [("Budapest X. Kőbánya", df_k_pontos), ("Bécs II. Leopoldstadt", df_l_pontos)]:
    if 'fajlagos_ar' in df.columns:
        norm_rows.append(compute_normality_metrics(df['fajlagos_ar'], 'Fajlagos ár [HUF/m² v. EUR/m²]', city))
    if 'ln_fajlagos_ar' in df.columns:
        norm_rows.append(compute_normality_metrics(df['ln_fajlagos_ar'], 'Log-fajlagos ár [ln(ár/m²)]', city))

df_normality_results = pd.DataFrame(norm_rows)
export_table(df_normality_results, "3_normalitas_tesztek", "Áreloszlások és logaritmizálás normalitásvizsgálata")


## 6. Modul: Árak Térbeli Megoszlása Távolsági Sávok Szerint

### Módszertani felépítés:
A folytonos távolságok diszkrét sávokba sorolása lehetővé teszi a nemlineáris infrastrukturális hatások és a zajsávok szerinti árszintek összehasonlítását.
- **Vasút sín távolsági kategóriák:** $0-150\text{m}$, $150-300\text{m}$, $300-500\text{m}$, $>500\text{m}$.
- **Vasútállomás távolsági kategóriák:** $0-375\text{m}$, $375-750\text{m}$, $750-1125\text{m}$, $>1125\text{m}$.

A modul kiszámítja az egyes sávok mintanagyságát ($N$), az átlagos fajlagos árat, szórást, mediánt és interkvartilis terjedelmet (IQR).


In [ ]:
# ==============================================================================
# MODUL 6: TÁVOLSÁGI SÁVOK SZERINTI ÁRSTATISZTIKÁK SZÁMÍTÁSA
# ==============================================================================
def assign_distance_zones(df: pd.DataFrame) -> pd.DataFrame:
    df_c = df.copy()

    def get_rail_band(r):
        val = r.get('dist_vagany_m')
        if pd.isna(val) or val == '': return np.nan
        d = float(val)
        if d <= 150: return '1. 0 - 150 m'
        elif d <= 300: return '2. 150 - 300 m'
        elif d <= 500: return '3. 300 - 500 m'
        else: return '4. > 500 m (Referencia)'

    def get_station_band(r):
        val = r.get('dist_vasutallomas_hm')
        if pd.isna(val) or val == '': return np.nan
        d = float(val)
        if d <= 375: return '1. 0 - 375 m'
        elif d <= 750: return '2. 375 - 750 m'
        elif d <= 1125: return '3. 750 - 1125 m'
        else: return '4. > 1125 m (Referencia)'

    df_c['rail_band'] = df_c.apply(get_rail_band, axis=1)
    df_c['station_band'] = df_c.apply(get_station_band, axis=1)
    return df_c

def compute_band_breakdown(df: pd.DataFrame, band_col: str, city_name: str, band_type: str) -> pd.DataFrame:
    df_valid = df.dropna(subset=[band_col]).copy()
    df_valid['fajlagos_ar'] = pd.to_numeric(df_valid['fajlagos_ar'], errors='coerce')
    df_valid = df_valid.dropna(subset=['fajlagos_ar'])

    rows = []
    for band_val, grp in df_valid.groupby(band_col):
        s_price = grp['fajlagos_ar']
        s_ln = pd.to_numeric(grp.get('ln_fajlagos_ar', np.log(s_price)), errors='coerce')
        rows.append({
            'Város': city_name,
            'Sáv Típus': band_type,
            'Távolsági Kategória': band_val,
            'N': len(grp),
            'Átlagos Fajlagos Ár': round(float(s_price.mean()), 2),
            'Szórás': round(float(s_price.std()), 2) if len(grp) > 1 else 0.0,
            'Medián Ár': round(float(s_price.median()), 2),
            'IQR (Q3 - Q1)': round(float(s_price.quantile(0.75) - s_price.quantile(0.25)), 2),
            'Átlag ln(ár/m²)': round(float(s_ln.mean()), 4)
        })
    return pd.DataFrame(rows)

df_k_zoned = assign_distance_zones(df_k_pontos)
df_l_zoned = assign_distance_zones(df_l_pontos)

rail_breakdown = pd.concat([
    compute_band_breakdown(df_k_zoned, 'rail_band', 'Budapest X. Kőbánya', 'Vasúti vágány távolság'),
    compute_band_breakdown(df_l_zoned, 'rail_band', 'Bécs II. Leopoldstadt', 'Vasúti vágány távolság')
], ignore_index=True)

station_breakdown = pd.concat([
    compute_band_breakdown(df_k_zoned, 'station_band', 'Budapest X. Kőbánya', 'Vasútállomás távolság'),
    compute_band_breakdown(df_l_zoned, 'station_band', 'Bécs II. Leopoldstadt', 'Vasútállomás távolság')
], ignore_index=True)

export_table(rail_breakdown, "4_arak_vasuti_savok_szerint", "Fajlagos árak statisztikája vasúti zajsávok szerint")
export_table(station_breakdown, "5_arak_allomas_savok_szerint", "Fajlagos árak statisztikája vasútállomási sávok szerint")

## 7. Modul: Korrelációs Mátrix és Multikollinearitás Ellenőrzése

### Matematikai alapok és küszöbértékek:
A regressziós becslés megbízhatóságát veszélyezteti a független változók közötti multikollinearitás:
- **Pearson-féle lineáris korrelációs együttható:**
  $$r_{jk} = \frac{\sum_{i=1}^N (x_{ij} - \bar{x}_j)(x_{ik} - \bar{x}_k)}{\sqrt{\sum_{i=1}^N (x_{ij} - \bar{x}_j)^2 \sum_{i=1}^N (x_{ik} - \bar{x}_k)^2}}$$
- **Variance Inflation Factor (Variancia Felfúvódási Tényező):**
  $$VIF_j = \frac{1}{1 - R_j^2}$$
  ahol $R_j^2$ az a determinációs együttható, amelyet a $j$-edik magyarázó változó összes többi magyarázó változóra való regressziójából kapunk.
- **Kritikus küszöbértékek:** $|r| > 0.70$ szoros együttmozgást jelez; $VIF > 5$ mérsékelt, $VIF > 10$ súlyos multikollinearitást mutat.


In [ ]:
# ==============================================================================
# MODUL 7: KORRELÁCIÓS MÁTRIX ÉS VIF SZÁMÍTÁSA (LOGARITMIKUS TÁVOLSÁGOKKAL)
# ==============================================================================
from statsmodels.stats.outliers_influence import variance_inflation_factor
import statsmodels.api as sm

CORR_FEATURES = [
    'ln_fajlagos_ar', 'alapterulet_nm', 'szobak_szama', 'd_felujitott',
    'ln_dist_fout_m', 'ln_dist_vagany_m', 'ln_dist_vasutallomas_hm',
    'd_lift', 'd_kulteri_kapcsolat', 'd_panel', 'd_epult_2015_utan'
]

FEATURE_LABELS = {
    'ln_fajlagos_ar': 'ln(Ár/m²)',
    'alapterulet_nm': 'Alapterület',
    'szobak_szama': 'Szobaszám',
    'd_felujitott': 'Felújított',
    'ln_dist_fout_m': 'ln(Főút táv.)',
    'ln_dist_vagany_m': 'ln(Sín táv.)',
    'ln_dist_vasutallomas_hm': 'ln(Állomás táv.)',
    'd_lift': 'Lift',
    'd_kulteri_kapcsolat': 'Erkély',
    'd_panel': 'Panel',
    'd_epult_2015_utan': 'Új építés'
}

def compute_correlation_and_vif(df: pd.DataFrame, city_name: str) -> tuple[pd.DataFrame, pd.DataFrame]:
    # Csak azokat a változókat korreláltatjuk, amelyek jelen vannak és szórásuk nem nulla (konstans oszlopok kiszűrése)
    valid_cols = [c for c in CORR_FEATURES if c in df.columns and df[c].std() > 0]
    df_sub = df[valid_cols].dropna()

    # Korrelációs mátrix (NaN mentes)
    corr_df = df_sub.corr().round(4)
    corr_df.index = [FEATURE_LABELS.get(c, c) for c in corr_df.index]
    corr_df.columns = [FEATURE_LABELS.get(c, c) for c in corr_df.columns]

    # VIF számítás a magyarázó változókra (csak azokra, amelyek varianciája > 0)
    exog_cols = [c for c in valid_cols if c != 'ln_fajlagos_ar' and df_sub[c].nunique() > 1]
    X = df_sub[exog_cols].astype(float)
    X_const = sm.add_constant(X, has_constant='add')

    vif_rows = []
    for i, col in enumerate(exog_cols):
        try:
            vif_val = variance_inflation_factor(X_const.values, i + 1)
        except Exception:
            vif_val = np.nan

        diag = 'Alacsony (VIF < 5)' if (not np.isnan(vif_val) and vif_val < 5) else (
            'Mérsékelt (5-10)' if (not np.isnan(vif_val) and vif_val < 10) else 'Súlyos (VIF > 10)'
        )
        vif_rows.append({
            'Város': city_name,
            'Változó': FEATURE_LABELS.get(col, col),
            'VIF Érték': round(float(vif_val), 3) if not np.isnan(vif_val) else np.nan,
            'Diagnózis': diag
        })

    vif_df = pd.DataFrame(vif_rows)
    return corr_df, vif_df

k_corr, k_vif = compute_correlation_and_vif(df_k_pontos, "Budapest X. Kőbánya")
l_corr, l_vif = compute_correlation_and_vif(df_l_pontos, "Bécs II. Leopoldstadt")

export_table(k_corr.reset_index(), "6_korrelacio_kobanya", "Korrelációs mátrix – Budapest X. Kőbánya")
export_table(l_corr.reset_index(), "6_korrelacio_leopoldstadt", "Korrelációs mátrix – Bécs II. Leopoldstadt")
export_table(pd.concat([k_vif, l_vif], ignore_index=True), "7_multikollinearitas_vif", "Multikollinearitás VIF vizsgálat (Hedonikus log-távolság specifikáció)")


# ==============================================================================
# 2. FŐRÉSZ: HEDONIKUS REGRESSZIÓS ELEMZÉS (OLS)
# ==============================================================================


## 3. Modul: A Négy Szimmetrikus Hedonikus Regressziós Modell Specifikációs Regisztere

### Módszertani felépítés és specifikációs családok:
Minden modell a log-fajlagos árat magyarázza: $\ln(\text{fajlagos\_ar}_i)$:
1. **M1 – Fizikai Alapmodell:** Alapvető ingatlanfizikai és minőségi jellemzők (alapterület, szobaszám, felújított állapot, lift, erkély/terasz, panel, szuterén, földszint, zárószint, 2015 után épült).
2. **M2 – Diszkrét Sávos Modell:** M1 + vasúti zajzónák ($0-150\text{m}, 150-300\text{m}, 300-500\text{m}$), vasútállomás gyalogos zónák ($0-375\text{m}, 375-750\text{m}, 750-1125\text{m}$) és főútvonal zajzóna ($0-150\text{m}$).
3. **M3 – Folytonos Logaritmikus Modell:** M1 + folytonos log-távolságok: $\ln(\text{dist\_vagany})$, $\ln(\text{dist\_vasutallomas})$ és $\ln(\text{dist\_fout})$.
4. **M4 – Fő Modell (Kettős Cenzorált Közelség + TOD Elérhetőség):** M1 + mindkét közlekedési zajexternália akusztikailag lecsengő cenzorált távolságfüggvénye (*Andersson et al. 2010; Day et al. 2007*):
   $$\text{track\_prox\_censored} = \max\left(0, \, \ln\left(\frac{300}{\text{dist\_vagany}}\right)\right)$$
   $$\text{road\_prox\_censored} = \max\left(0, \, \ln\left(\frac{150}{\text{dist\_fout}}\right)\right)$$
   kiegészítve a vasútállomás folytonos gyalogos TOD elérhetőségével: $\ln(\text{dist\_vasutallomas\_hm})$.

A modellek az alábbi `HEDONIC_SPECS` konfigurációs szótárban ellenőrizhetők és futtathatók.

In [ ]:
# ==============================================================================
# MODUL 3: REGISZTER ÉS AZ UNIVERZÁLIS HEDONIKUS BECSLŐ MOTOR (HC3)
# ==============================================================================
BASE_VARS = [
    'alapterulet_nm', 'szobak_szama', 'd_felujitando', 'd_felujitott',
    'd_lift', 'd_kulteri_kapcsolat', 'kulteri_alapterulet_nm', 'd_panel',
    'd_szuteren', 'd_foldszint', 'd_zaroszint', 'd_epult_2015_utan'
]

VARIABLE_LABELS = {
    'const': 'Konstans (Intercept)',
    'alapterulet_nm': 'Alapterület (m²)',
    'szobak_szama': 'Szobák száma (db)',
    'd_felujitott': 'Felújított állapot dummy',
    'd_felujitando': 'Felújítandó állapot dummy',
    'd_lift': 'Lift jelenléte dummy',
    'd_kulteri_kapcsolat': 'Erkély / terasz kapcsolat dummy',
    'kulteri_alapterulet_nm': 'Kültéri méret (erkély/terasz m²)',
    'd_panel': 'Panel építési technológia dummy',
    'd_szuteren': 'Szuterén szint dummy',
    'd_foldszint': 'Földszinti elhelyezkedés dummy',
    'd_zaroszint': 'Zárószint / legfelső emelet dummy',
    'd_epult_2015_utan': 'Új építés (2015 után) dummy',
    'd_vagany_0_150': 'Vasút sín [0 - 150 m] dummy',
    'd_vagany_150_300': 'Vasút sín [150 - 300 m] dummy',
    'd_vagany_300_500': 'Vasút sín [300 - 500 m] dummy',
    'd_vasutallomas_0_375': 'Vasútállomás [0 - 375 m] dummy',
    'd_vasutallomas_375_750': 'Vasútállomás [375 - 750 m] dummy',
    'd_vasutallomas_750_1125': 'Vasútállomás [750 - 1125 m] dummy',
    'd_fout_0_150': 'Főút közelség [0 - 150 m] dummy',
    'ln_dist_vagany_m': 'ln(Vasúti sín távolság m)',
    'track_prox_censored': 'Cenzorált vasúti közelség max(0, ln(300/dist))',
    'station_access_censored_1125': 'Cenzorált állomástávolság min(0, ln(dist/1125))',
    'station_access_censored_1125': '15-perces állomáselérhetőség max(0, ln(1125/dist))',
    'd_vasutallomas_0_750': 'Vasútállomás [0 - 750 m] dummy',
    'dist_vagany_m': 'Vasúti sín távolság (m)',
    'ln_dist_vasutallomas_hm': 'ln(Vasútállomás távolság m)',
    'dist_vasutallomas_hm': 'Vasútállomás távolság (m)',
    'ln_dist_fout_m': 'ln(Főút távolság m)',
    'dist_fout_m': 'Főút távolság (m)'
}

# 4 fókuszált tudományos modellcsalád:
HEDONIC_SPECS = {
    'M1': {
        'name': 'M1: Alapmodell (Fizikai ingatlantulajdonságok)',
        'category': 'Bázismodell',
        'features': BASE_VARS.copy()
    },
    'M2': {
        'name': 'M2: Diszkrét távolsági sávos modell (Zaj- és állomás-pufferek)',
        'category': 'Diszkrét sávos',
        'features': BASE_VARS + ['d_vagany_0_150', 'd_vagany_150_300', 'd_vagany_300_500',
                                 'd_vasutallomas_0_375', 'd_vasutallomas_375_750', 'd_vasutallomas_750_1125', 'd_fout_0_150']
    },
    'M3': {
        'name': 'M3: Folytonos logaritmikus modell (ln-távolságok)',
        'category': 'Folytonos logaritmikus',
        'features': BASE_VARS + ['ln_dist_vagany_m', 'ln_dist_vasutallomas_hm', 'ln_dist_fout_m']
    },
    'M4': {
        'name': 'M4: Hármas cenzorált zászlóshajó modell (Vasút 300m, Főút 150m, Állomás 1125m)',
        'category': 'Cenzorált közelségi',
        'features': BASE_VARS + ['track_prox_censored', 'road_prox_censored', 'station_access_censored_1125']
    }
}

def estimate_single_hedonic_model(spec_key: str, spec: dict, df: pd.DataFrame, cov_type: str = 'HC3'):
    features = spec['features'].copy()
    # Dinamikus kategória-összevonás (pl. Bécs M2-ben N=1 miatti SE robbanás elkerülésére):
    if spec_key == 'M2' and 'd_vasutallomas_0_375' in features and 'd_vasutallomas_375_750' in features:
        if 'd_vasutallomas_0_375' in df.columns and df['d_vasutallomas_0_375'].sum() < 5 and 'd_vasutallomas_0_750' in df.columns:
            features = [f for f in features if f not in ['d_vasutallomas_0_375', 'd_vasutallomas_375_750']]
            features.insert(features.index('d_vasutallomas_750_1125'), 'd_vasutallomas_0_750')

    req_cols = ['ln_fajlagos_ar'] + features
    # Csak azokat a változókat tartjuk meg, amelyek léteznek és van varianciájuk az adott mintában
    valid_cols = [c for c in req_cols if c in df.columns and (c == 'ln_fajlagos_ar' or df[c].nunique() > 1)]

    df_clean = df[valid_cols].dropna()

    y = df_clean['ln_fajlagos_ar'].astype(float)
    X = df_clean[[c for c in valid_cols if c != 'ln_fajlagos_ar']].astype(float)
    X = sm.add_constant(X, has_constant='add')

    model = sm.OLS(y, X)
    res = model.fit(cov_type=cov_type)

    param_rows = []
    for var_name in X.columns:
        c = res.params[var_name]
        se = res.bse[var_name]
        t = res.tvalues[var_name]
        p = res.pvalues[var_name]
        ci = res.conf_int().loc[var_name]
        param_rows.append({
            'Modell': spec_key,
            'Változó Kód': var_name,
            'Változó Megnevezés': VARIABLE_LABELS.get(var_name, var_name),
            'Együttható (Beta)': round(float(c), 4),
            'Robusztus SE (HC3)': round(float(se), 4),
            't-érték': round(float(t), 3),
            'p-érték': round(float(p), 4),
            'Szignifikancia': '***' if p < 0.01 else ('**' if p < 0.05 else ('*' if p < 0.1 else '')),
            '95% CI Alsó': round(float(ci[0]), 4),
            '95% CI Felső': round(float(ci[1]), 4)
        })

    metrics = {
        'Modell': spec_key,
        'N (Megfigyelések)': int(res.nobs),
        'K (Paraméterek)': int(len(X.columns)),
        'R²': round(float(res.rsquared), 4),
        'Korrigált R²': round(float(res.rsquared_adj), 4),
        'F-statisztika': round(float(res.fvalue), 2) if res.fvalue is not None else np.nan,
        'F p-érték': round(float(res.f_pvalue), 5) if res.f_pvalue is not None else np.nan,
        'Log-Likelihood': round(float(res.llf), 2),
        'AIC': round(float(res.aic), 2),
        'BIC': round(float(res.bic), 2)
    }

    return pd.DataFrame(param_rows), metrics, res


## 4. Modul: Budapest és Bécs Hedonikus Modelljeinek Becslése

### Eredménytáblázat specifikációja:
A becslések az összes specifikációra (M1–M11) lefuttathatók külön-külön vagy kötegelten mindkét város mintájára. A kimeneti nagytáblázat tartalmazza a becsült együtthatókat, robusztus standard hibákat ($HC3$) és a modellek leíró illeszkedési statisztikáit.


In [ ]:
# ==============================================================================
# MODUL 4: HEDONIKUS MODELLEK KÖTEGELT BECSLÉSE ÉS NAGYTÁBLÁZAT GENERÁLÁSA
# ==============================================================================
def estimate_all_hedonic_models(df: pd.DataFrame, city_name: str, sample_name: str):
    all_params = []
    all_metrics = []
    fitted_models = {}

    for m_code, spec in HEDONIC_SPECS.items():
        df_p, m_dict, res_obj = estimate_single_hedonic_model(m_code, spec, df, cov_type='HC3')
        df_p['Város'] = city_name
        df_p['Minta'] = sample_name
        m_dict['Város'] = city_name
        m_dict['Minta'] = sample_name
        m_dict['Specifikáció'] = spec['name']
        m_dict['Kategória'] = spec['category']

        all_params.append(df_p)
        all_metrics.append(m_dict)
        fitted_models[m_code] = res_obj

    df_all_params = pd.concat(all_params, ignore_index=True)
    df_all_metrics = pd.DataFrame(all_metrics)
    log_info(f"{city_name} [{sample_name}]: Mind a 11 hedonikus modell sikeresen megbecsülve (HC3).")
    return df_all_params, df_all_metrics, fitted_models

params_k, metrics_k, models_k = estimate_all_hedonic_models(df_k_pontos, "Budapest X. Kőbánya", "Pontos")
params_l, metrics_l, models_l = estimate_all_hedonic_models(df_l_pontos, "Bécs II. Leopoldstadt", "Pontos")

export_table(metrics_k, "8_hedonikus_modellek_osszesites_kobanya", "Hedonikus modellek illeszkedési mutatói – Budapest X. Kőbánya")
export_table(metrics_l, "8_hedonikus_modellek_osszesites_leopoldstadt", "Hedonikus modellek illeszkedési mutatói – Bécs II. Leopoldstadt")
export_table(pd.concat([params_k, params_l], ignore_index=True), "9_hedonikus_egyutthatok_minden_modell", "Minden hedonikus modell részletes együtthatói és robusztus SE-je")


## 5. Modul: Ökonometriai Modellválasztási Elemzés (Model Selection)

### Matematikai információs kritériumok:
Az alternatív funkcionális formák (diszkrét sávos vs. folytonos log-rugalmassági vs. hibrid) összehasonlítása az alábbi kritériumok alapján történik:
- **Korrigált $R^2$:** $\bar{R}^2 = 1 - (1 - R^2) \frac{n - 1}{n - k - 1}$
- **Akaike Információs Kritérium:** $\text{AIC} = 2k - 2\ln(\hat{L})$
- **Bayes-féle / Schwarz Információs Kritérium:** $\text{BIC} = k\ln(n) - 2\ln(\hat{L})$

A legkedvezőbb, leginkább parszimóniás modellt a minimális AIC és BIC érték jelöli ki.


In [ ]:
# ==============================================================================
# MODUL 5: ÖSSZEHASONLÍTÓ MODELLVÁLASZTÁSI TÁBLÁZAT
# ==============================================================================
def build_model_selection_comparison(metrics_k: pd.DataFrame, metrics_l: pd.DataFrame) -> pd.DataFrame:
    df_merged = metrics_k[['Modell', 'Kategória', 'Specifikáció', 'Korrigált R²', 'AIC', 'BIC']].copy()
    df_merged.columns = ['Modell', 'Kategória', 'Specifikáció', 'Kőbánya Adj-R²', 'Kőbánya AIC', 'Kőbánya BIC']

    l_sub = metrics_l[['Modell', 'Korrigált R²', 'AIC', 'BIC']].copy()
    l_sub.columns = ['Modell', 'Leopoldstadt Adj-R²', 'Leopoldstadt AIC', 'Leopoldstadt BIC']

    res = pd.merge(df_merged, l_sub, on='Modell')
    return res

df_model_selection = build_model_selection_comparison(metrics_k, metrics_l)
export_table(df_model_selection, "10_modellvalasztas_aic_bic", "Ökonometriai modellválasztási táblázat (AIC, BIC, Adj-R²)")


## 6. Modul: Ökonometriai Diagnosztikai Tesztcsomag

### Alkalmazott tesztek és nullhipotéziseik ($H_0$):
1. **Multikollinearitás:** Variancia Felfúvódási Tényező ($VIF_j = \frac{1}{1 - R_j^2}$), kritikus küszöb: $VIF < 5$.
2. **Breusch-Pagan teszt (Heteroszkedaszticitás):** $LM = \frac{1}{2} ESS \sim \chi^2(k)$. Nullhipotézis $H_0$: A hibatagok varianciája homoszkedasztikus (állandó).
3. **White-féle általános heteroszkedaszticitási teszt:** Keresztszorzatokat tartalmazó kiegészítő regresszió. $H_0$: Homoszkedaszticitás.
4. **Ramsey RESET specifikációs teszt:** $y = X\beta + \sum_{p=2}^P \gamma_p \hat{y}^p + u$. Nullhipotézis $H_0$: $\gamma_2 = \dots = \gamma_P = 0$ (nincsenek kihagyott nemlineáris változók, a funkcionális forma helyes).
5. **Jarque-Bera normalitási teszt:** $JB = \frac{N}{6} \left( S^2 + \frac{(K-3)^2}{4} \right) \sim \chi^2(2)$. Nullhipotézis $H_0$: A regressziós reziduumok normális eloszlásúak.


In [ ]:
# ==============================================================================
# MODUL 6: ÖKONOMETRIAI DIAGNOSZTIKAI CSOMAG FUTTATÁSA
# ==============================================================================
KEY_DIAGNOSTIC_MODELS = ['M1', 'M2', 'M3', 'M4']

def run_econometric_diagnostics(model_dict: dict, city_name: str) -> pd.DataFrame:
    rows = []
    for m_code in KEY_DIAGNOSTIC_MODELS:
        if m_code not in model_dict: continue
        res = model_dict[m_code]
        model = res.model
        X = model.exog
        y = model.endog
        resid = res.resid

        # 1. Max VIF
        vif_vals = [variance_inflation_factor(X, i) for i in range(1, X.shape[1])]
        max_vif = max(vif_vals) if vif_vals else 1.0

        # 2. Breusch-Pagan
        bp_lm, bp_p, _, _ = smd.het_breuschpagan(resid, X)

        # 3. White teszt
        try:
            wh_lm, wh_p, _, _ = smd.het_white(resid, X)
        except Exception:
            wh_lm, wh_p = np.nan, np.nan

        # 4. Ramsey RESET (linear_reset használata a HC3 kompatibilitásért)
        try:
            from statsmodels.stats.diagnostic import linear_reset
            reset_res = linear_reset(res, power=3, use_f=True, cov_type='HC3')
            reset_f = float(reset_res.fvalue)
            reset_p = float(reset_res.pvalue)
        except Exception:
            try:
                reset_res = smd.reset_ramsey(res, degree=3)
                reset_f = float(np.squeeze(reset_res.fvalue))
                reset_p = float(reset_res.pvalue)
            except Exception:
                reset_f, reset_p = np.nan, np.nan

        # 5. Jarque-Bera
        jb_val, jb_p = stats.jarque_bera(resid)

        rows.append({
            'Város': city_name,
            'Modell': m_code,
            'Specifikáció': HEDONIC_SPECS[m_code]['name'],
            'Max VIF': round(float(max_vif), 2),
            'VIF Státusz': 'Rendben (<5)' if max_vif < 5 else 'Multikollinearitás',
            'Breusch-Pagan LM': round(float(bp_lm), 2),
            'BP p-érték': round(float(bp_p), 5),
            'White LM': round(float(wh_lm), 2) if not np.isnan(wh_lm) else np.nan,
            'White p-érték': round(float(wh_p), 5) if not np.isnan(wh_p) else np.nan,
            'Ramsey RESET F': round(float(reset_f), 2) if not np.isnan(reset_f) else np.nan,
            'RESET p-érték': round(float(reset_p), 5) if not np.isnan(reset_p) else np.nan,
            'Jarque-Bera Stat': round(float(jb_val), 2),
            'JB p-érték': round(float(jb_p), 5)
        })
    return pd.DataFrame(rows)

df_diag_k = run_econometric_diagnostics(models_k, "Budapest X. Kőbánya")
df_diag_l = run_econometric_diagnostics(models_l, "Bécs II. Leopoldstadt")
df_diagnostics = pd.concat([df_diag_k, df_diag_l], ignore_index=True)

export_table(df_diagnostics, "11_okonometriai_diagnosztika", "Ökonometriai diagnosztikai teszteredmények (VIF, BP, White, RESET, JB)")


## 7. Modul: Robusztussági Vizsgálatok (Heckman Szelekció és Klaszterezett Standard Hibák)

### Matematikai specifikáció:
1. **Heckman-féle minta-szelekciós korrekció (Heckman 1979):**
   A pontos geokódolású almintába kerülés valószínűségét probit modell becsüli:
   $$P(\text{Pontos}_i = 1 \mid Z_i) = \Phi(Z_i' \gamma)$$
   Ebből képzett Inverz Mills-arány (IMR):
   $$\lambda_i = \frac{\phi(Z_i' \hat{\gamma})}{\Phi(Z_i' \hat{\gamma})}$$
   A hedonikus regresszió kiegészül a $\lambda_i$ regresszorral: $\ln(P_i) = X_i' \beta + \rho \sigma_u \lambda_i + v_i$.
   $H_0: \beta_{\lambda} = 0$ (nincs szelekciós torzítás a pontos minta szűkítéséből).
2. **Utcaszintű klaszterezett standard hibák (Clustered Standard Errors):**
   Azonos utcában lévő lakások megfigyeletlen korrelációjának kezelése:
   $$\hat{\mathbf{V}}_{\text{cluster}} = (\mathbf{X}'\mathbf{X})^{-1} \left( \sum_{g=1}^G \mathbf{X}_g' \mathbf{e}_g \mathbf{e}_g' \mathbf{X}_g \right) (\mathbf{X}'\mathbf{X})^{-1}$$


In [ ]:
# ==============================================================================
# MODUL 7: ROBUSZTUSSÁGI VIZSGÁLAT (2-LÉPCSŐS HECKMAN ÉS VALÓS UTCAI KLASZTEREZÉS)
# ==============================================================================
import re

def extract_street_clean(addr_str: str) -> str:
    s = str(addr_str).strip()
    # Eltávolítjuk a házszámot és az utána következő szöveget
    s_clean = re.sub(r'\s+\d+.*$', '', s)
    return s_clean if s_clean else s

def run_heckman_and_cluster_diagnostics(df_full: pd.DataFrame, df_pontos: pd.DataFrame, city_name: str) -> pd.DataFrame:
    # A benchmark specifikáció a folytonos log-távolsági M3 modell
    m3_features = [c for c in HEDONIC_SPECS['M3']['features'] if c in df_pontos.columns and df_pontos[c].nunique() > 1]
    df_clean = df_pontos[['ln_fajlagos_ar', 'cim_teljes'] + m3_features].dropna().copy()

    y = df_clean['ln_fajlagos_ar'].astype(float)
    X = sm.add_constant(df_clean[m3_features].astype(float))

    # 1. Alap HC3 OLS modell
    ols_hc3 = sm.OLS(y, X).fit(cov_type='HC3')

    # 2. Valós utcaszintű klaszterezett standard hibák (Clustered SE)
    cluster_group = df_clean['cim_teljes'].apply(extract_street_clean)
    ols_clust = sm.OLS(y, X).fit(cov_type='cluster', cov_kwds={'groups': cluster_group})

    # 3. Klasszikus 2-lépcsős Heckman szelekciós modell (Probit 1. lépés a teljes mintán)
    df_full_copy = df_full.copy()
    df_full_copy['is_geocoded'] = df_full_copy['wgs84_lat'].notnull().astype(int)

    probit_vars = ['alapterulet_nm', 'szobak_szama', 'd_lift']
    if 'd_panel' in df_full_copy.columns and df_full_copy['d_panel'].nunique() > 1:
        probit_vars.append('d_panel')

    df_probit = df_full_copy[probit_vars + ['is_geocoded']].dropna().copy()
    X_probit = sm.add_constant(df_probit[probit_vars].astype(float))
    probit_res = sm.Probit(df_probit['is_geocoded'], X_probit).fit(disp=False)

    # Inverz Mills-arány (IMR) kiszámítása a geokódolt almintára
    xb = probit_res.predict(which='linear')
    imr_series = stats.norm.pdf(xb) / np.maximum(1e-4, stats.norm.cdf(xb))
    df_full_copy['lambda_imr'] = np.nan
    df_full_copy.loc[df_probit.index, 'lambda_imr'] = imr_series

    # Explicit listing_id alapú baloldali összekapcsolás (merge) a pontos mintához
    if 'listing_id' in df_clean.columns and 'listing_id' in df_full_copy.columns:
        df_clean = df_clean.merge(df_full_copy[['listing_id', 'lambda_imr']], on='listing_id', how='left')
    else:
        df_clean['lambda_imr'] = df_full_copy.loc[df_clean.index, 'lambda_imr']
    df_clean['lambda_imr'] = df_clean['lambda_imr'].fillna(float(imr_series.mean()))

    X_heck = sm.add_constant(df_clean[m3_features + ['lambda_imr']].astype(float))
    ols_heck = sm.OLS(y, X_heck).fit(cov_type='HC3')

    rows = []
    target_vars = ['ln_dist_vagany_m', 'ln_dist_vasutallomas_hm', 'ln_dist_fout_m']
    for v in target_vars:
        if v in X.columns:
            rows.append({
                'Város': city_name,
                'Változó': VARIABLE_LABELS.get(v, v),
                'Alap OLS (HC3) Beta': round(float(ols_hc3.params[v]), 4),
                'Alap HC3 SE': round(float(ols_hc3.bse[v]), 4),
                'Alap HC3 p-érték': round(float(ols_hc3.pvalues[v]), 4),
                'Klaszterezett SE': round(float(ols_clust.bse[v]), 4),
                'Klaszterezett p-érték': round(float(ols_clust.pvalues[v]), 4),
                'Heckman IMR Korrigált Beta': round(float(ols_heck.params[v]), 4),
                'Heckman IMR SE': round(float(ols_heck.bse[v]), 4),
                'Heckman IMR p-érték': round(float(ols_heck.pvalues[v]), 4)
            })

    # IMR Lambda együttható sora (szelekciós torzítás szignifikanciája)
    if 'lambda_imr' in ols_heck.params:
        rows.append({
            'Város': city_name,
            'Változó': 'Inverz Mills-arány (Lambda IMR)',
            'Alap OLS (HC3) Beta': np.nan,
            'Alap HC3 SE': np.nan,
            'Alap HC3 p-érték': np.nan,
            'Klaszterezett SE': np.nan,
            'Klaszterezett p-érték': np.nan,
            'Heckman IMR Korrigált Beta': round(float(ols_heck.params['lambda_imr']), 4),
            'Heckman IMR SE': round(float(ols_heck.bse['lambda_imr']), 4),
            'Heckman IMR p-érték': round(float(ols_heck.pvalues['lambda_imr']), 4)
        })
    return pd.DataFrame(rows)

df_rob_k = run_heckman_and_cluster_diagnostics(df_kobanya, df_k_pontos, "Budapest X. Kőbánya")
df_rob_l = run_heckman_and_cluster_diagnostics(df_leopoldstadt, df_l_pontos, "Bécs II. Leopoldstadt")
df_robustness = pd.concat([df_rob_k, df_rob_l], ignore_index=True)

export_table(df_robustness, "12_robusztussag_heckman_cluster", "Robusztussági ellenőrzések (2-lépcsős Heckman IMR és Utcaszintű Klaszterezett SE)")


# ==============================================================================
# 3. FŐRÉSZ: TÉRBELI ÖKONOMETRIA (SAR & SEM)
# ==============================================================================


## 2. Modul: Adatbetöltés és Hivatalos Metrikus Vetületi Transzformációk

### Matematikai és geodéziai specifikáció:
A térbeli távolságok és súlymátrixok torzításmentes számításához a WGS84 földrajzi koordinátákat (fok) hivatalos metrikus síkvetületekbe kell transzformálni:
- **Budapest X. Kőbánya:** Egységes Országos Vetület (EOV / EPSG:23700).
- **Bécs II. Leopoldstadt:** MGI / Austria Lambert ill. Gauss-Krüger East (EPSG:31256).

Az euklideszi távolság a transzformált koordinátákon:
$$d_{ij} = \sqrt{(x_i - x_j)^2 + (y_i - y_j)^2} \quad [\text{méterben}]$$


In [ ]:
# ==============================================================================
# MODUL 2: TÉRBELI VETÜLETI KOORDINÁTÁK ÉS GEODATAFRAME ELŐÁLLÍTÁSA
# ==============================================================================
def create_projected_spatial_dataframe(df: pd.DataFrame, city_label: str, target_epsg: int) -> gpd.GeoDataFrame:
    df_clean = df.copy()
    lon_col = 'lon' if 'lon' in df_clean.columns else ('lng' if 'lng' in df_clean.columns else 'wgs84_long')
    lat_col = 'lat' if 'lat' in df_clean.columns else 'wgs84_lat'

    df_clean = df_clean.dropna(subset=[lat_col, lon_col, 'ln_fajlagos_ar']).copy()
    geometry = [Point(xy) for xy in zip(df_clean[lon_col], df_clean[lat_col])]
    gdf_wgs84 = gpd.GeoDataFrame(df_clean, geometry=geometry, crs="EPSG:4326")
    gdf_proj = gdf_wgs84.to_crs(epsg=target_epsg)

    gdf_proj['x_proj'] = gdf_proj.geometry.x
    gdf_proj['y_proj'] = gdf_proj.geometry.y
    gdf_proj['u_proj'] = gdf_proj.geometry.x
    gdf_proj['v_proj'] = gdf_proj.geometry.y

    log_info(f"{city_label}: Metrikus síkvetületi réteg előállítva (EPSG:{target_epsg}, N={len(gdf_proj)})")
    return gdf_proj

gdf_k_pontos = create_projected_spatial_dataframe(df_k_pontos, "Budapest X. Kőbánya", target_epsg=23700)
gdf_l_pontos = create_projected_spatial_dataframe(df_l_pontos, "Bécs II. Leopoldstadt", target_epsg=31256)


## 3. Modul: Térbeli Súlymátrixok ($W$) Felépítése és Topológiai Ellenőrzése

### Matematikai specifikáció és standardizálás:
A térbeli kapcsolatok leírására három alternatív súlymátrix-topológiát hozunk létre:
1. **$k$-Legközelebbi Szomszéd ($k$-NN) – Elsődleges bázismátrix ($k=5$):**
   $$w_{ij}^* = \begin{cases} 1, & \text{ha } j \in N_k(i) \\ 0, & \text{egyébként} \end{cases} \quad (k=5 \text{ bázis, } k=8 \text{ robusztussági ellenőrzés})$$
2. **Inverz Távolság Súlyozás (Inverse Distance) – Robusztussági mátrix:**
   $$w_{ij}^* = \frac{1}{\max(1.0, d_{ij})} \quad (d_{ij} \le 2500\,\text{m})$$
   (Azonos koordinátájú ingatlanok/épületek esetén $d_{ij} = 0 \to \max(1.0, d) = 1.0\,\text{m}$, elkerülve a zéróosztást).
3. **Sorstandardizálás (Row-standardization):**
   $$w_{ij} = \frac{w_{ij}^*}{\sum_{j=1}^n w_{ij}^*}, \quad \sum_{j=1}^n w_{ij} = 1$$

A modellezés során a **$k$-NN 5 ($k=5$)** képezi a fő ökonometriai becslések bázisát (Moran's I, Anselin LM-fa, SAR, SEM, LeSage & Pace), míg a $k=8$ és az inverz távolság súlymátrixok a 10. modulban dedikált robusztussági tesztelésre kerülnek.

In [ ]:
# ==============================================================================
# MODUL 3: TÉRBELI SÚLYMÁTRIXOK (W) LÉTREHOZÁSA ÉS DIAGNOSZTIKÁJA
# ==============================================================================
from scipy.spatial.distance import pdist, squareform

def construct_spatial_weights(gdf: gpd.GeoDataFrame, city_label: str) -> dict:
    coords = np.array(list(zip(gdf['x_proj'], gdf['y_proj'])))

    # 1. k-NN súlyok (k=5 elsődleges bázis, k=8 alternatív)
    w_knn5 = weights.KNN.from_array(coords, k=5)
    w_knn5.transform = 'R'

    w_knn8 = weights.KNN.from_array(coords, k=8)
    w_knn8.transform = 'R'

    # 2. Inverz távolság súlymátrix (2500 m küszöb, azonos épület d=0 -> min 1.0 m)
    d_mat = squareform(pdist(coords))
    np.fill_diagonal(d_mat, np.inf)
    d_mat = np.maximum(1.0, d_mat)
    
    neighbors = {}
    w_values = {}
    for i in range(len(coords)):
        idx = np.where(d_mat[i] <= 2500.0)[0]
        neighbors[i] = list(idx)
        w_values[i] = list(1.0 / d_mat[i, idx])

    w_dist = weights.W(neighbors, w_values)
    w_dist.transform = 'R'

    log_info(f"{city_label}: Súlymátrixok (k-NN 5 [Bázis], k-NN 8, Inverz távolság) sikeresen felépítve.")
    return {
        'w_knn5': w_knn5,
        'w_knn8': w_knn8,
        'w_inv': w_dist
    }

w_dict_k = construct_spatial_weights(gdf_k_pontos, "Budapest X. Kőbánya")
w_dict_l = construct_spatial_weights(gdf_l_pontos, "Bécs II. Leopoldstadt")

def get_weights_summary(w_dict: dict, city_label: str) -> pd.DataFrame:
    rows = []
    for w_name, w_obj in w_dict.items():
        rows.append({
            'Város': city_label,
            'Súlymátrix Típus': w_name,
            'Megfigyelések száma (N)': w_obj.n,
            'Átlagos szomszédszám': round(float(w_obj.mean_neighbors), 2),
            'Min szomszédszám': int(w_obj.min_neighbors),
            'Max szomszédszám': int(w_obj.max_neighbors),
            'Szigetek (izolált pontok)': len(w_obj.islands),
            'Nemnulla elemek %': round(float(w_obj.pct_nonzero), 3)
        })
    return pd.DataFrame(rows)

df_weights_summary = pd.concat([
    get_weights_summary(w_dict_k, "Budapest X. Kőbánya"),
    get_weights_summary(w_dict_l, "Bécs II. Leopoldstadt")
], ignore_index=True)

export_table(df_weights_summary, "13_terbeli_sulymatrixok_osszesites", "Térbeli súlymátrixok összefoglaló topológiai statisztikái")

## 4. Modul: Globális Moran's I Térbeli Autokorrelációs Tesztek

### Matematikai specifikáció és nullhipotézis:
A globális Moran's $I$ együttható méri a térbeli autokorreláció jelenlétét a folytonos változókban:
$$I = \frac{n}{S_0} \frac{\sum_{i=1}^n \sum_{j=1}^n w_{ij} (y_i - \bar{y})(y_j - \bar{y})}{\sum_{i=1}^n (y_i - \bar{y})^2}, \quad S_0 = \sum_{i=1}^n \sum_{j=1}^n w_{ij}$$
- **Várható érték térbeli függetlenség esetén:** $E[I] = -\frac{1}{n - 1}$
- **Nullhipotézis ($H_0$):** A megfigyelt értékek térben véletlenszerűen oszlanak el (nincs térbeli klaszterezettség). $p < 0.05$ esetén a térbeli függetlenség elvetve ($I > E[I]$ pozitív, $I < E[I]$ negatív autokorrelációt jelez).

A tesztet elvégezzük a nyers árakra, valamint a négy bázis OLS modell (M3, M6, M7, M8) hibatagjaira is.


In [ ]:
# ==============================================================================
# MODUL 4: MORAN'S I TESZTEK SZÁMÍTÁSA (M1 - M4 HANGOLVA, k-NN 5 BÁZIS)
# ==============================================================================
SPATIAL_SPECS = {
    'M1': {
        'name': 'M1: Alapmodell (Fizikai)',
        'features': BASE_VARS.copy()
    },
    'M2': {
        'name': 'M2: Diszkrét sávos modell',
        'features': BASE_VARS + ['d_vagany_0_150', 'd_vagany_150_300', 'd_vagany_300_500',
                                 'd_vasutallomas_0_375', 'd_vasutallomas_375_750', 'd_vasutallomas_750_1125', 'd_fout_0_150']
    },
    'M3': {
        'name': 'M3: Folytonos log-modell',
        'features': BASE_VARS + ['ln_dist_vagany_m', 'ln_dist_vasutallomas_hm', 'ln_dist_fout_m']
    },
    'M4': {
        'name': 'M4: Hármas cenzorált zászlóshajó modell (1125m)',
        'features': BASE_VARS + ['track_prox_censored', 'road_prox_censored', 'station_access_censored_1125']
    }
}

def compute_morans_i_suite(gdf: gpd.GeoDataFrame, w_obj, city_label: str) -> tuple[pd.DataFrame, dict]:
    y = gdf['ln_fajlagos_ar'].values

    # 1. Nyers árak Moran I
    mi_raw = esda.moran.Moran(y, w_obj)
    rows = [{
        'Város': city_label,
        'Célváltozó': 'Nyers ln(ár/m²)',
        "Morans I": round(float(mi_raw.I), 4),
        'Várható érték E[I]': round(float(mi_raw.EI), 4),
        'z-érték': round(float(mi_raw.z_norm), 3),
        'p-érték': round(float(mi_raw.p_norm), 5),
        'H0 (Függetlenség)': 'Elvetve (Klaszterezett)' if mi_raw.p_norm < 0.05 else 'Elfogadva'
    }]

    ols_models = {}
    for m_code, spec in SPATIAL_SPECS.items():
        feat = spec['features'].copy()
        if m_code == 'M2' and 'd_vasutallomas_0_375' in feat and 'd_vasutallomas_375_750' in feat:
            if 'd_vasutallomas_0_375' in gdf.columns and gdf['d_vasutallomas_0_375'].sum() < 5 and 'd_vasutallomas_0_750' in gdf.columns:
                feat = [f for f in feat if f not in ['d_vasutallomas_0_375', 'd_vasutallomas_375_750']]
                feat.insert(feat.index('d_vasutallomas_750_1125'), 'd_vasutallomas_0_750')
        feat = [c for c in feat if c in gdf.columns and gdf[c].nunique() > 1]
        X = gdf[feat].values

        # spreg OLS becslés
        ols_m = spreg.OLS(y[:, None], X, w=w_obj, name_y='ln_fajlagos_ar', name_x=feat, spat_diag=True)
        ols_models[m_code] = ols_m

        # Reziduum Moran I
        mi_res = esda.moran.Moran(ols_m.u.flatten(), w_obj)
        rows.append({
            'Város': city_label,
            'Célváltozó': f'{m_code} OLS reziduum',
            "Morans I": round(float(mi_res.I), 4),
            'Várható érték E[I]': round(float(mi_res.EI), 4),
            'z-érték': round(float(mi_res.z_norm), 3),
            'p-érték': round(float(mi_res.p_norm), 5),
            'H0 (Függetlenség)': 'Elvetve (Reziduum autokorr.)' if mi_res.p_norm < 0.05 else 'Elfogadva'
        })

    return pd.DataFrame(rows), ols_models

df_moran_k, ols_k = compute_morans_i_suite(gdf_k_pontos, w_dict_k['w_knn5'], "Budapest X. Kőbánya")
df_moran_l, ols_l = compute_morans_i_suite(gdf_l_pontos, w_dict_l['w_knn5'], "Bécs II. Leopoldstadt")
df_moran_summary = pd.concat([df_moran_k, df_moran_l], ignore_index=True)

export_table(df_moran_summary, "14_moran_i_tesztek", "Globális Moran's I térbeli autokorrelációs tesztek (Árak és OLS reziduumok)")

## 5. Modul: Anselin (1988) Lagrange Multiplier (LM) Diagnosztikai Döntési Fa

### Döntési algoritmus specifikációja:
A klasszikus OLS becslés és a térbeli folyamatok közötti specifikációválasztás az Anselin-féle LM tesztek alapján történik ($\chi^2(1)$ aszimptotikus eloszlással):
1. **Egyszerű LM tesztek:**
   - $	ext{LM-Lag}$ ($H_0$: $
ho = 0$, nincs térbeli késleltetett endogén hatás)
   - $	ext{LM-Error}$ ($H_0$: $\lambda = 0$, nincs térbeli autokorreláció a hibatagban)
2. **Robusztus LM tesztek:**
   Amennyiben mindkét egyszerű LM teszt szignifikáns ($p < 0.05$), a helyi robusztus verziók döntenek:
   - $	ext{RLM-Lag}$ (LM-Lag korrigálva az esetleges térbeli hibahatásra)
   - $	ext{RLM-Error}$ (LM-Error korrigálva az esetleges térbeli késleltetett hatásra)
3. **Optimális modell kijelölése:**
   - Ha egyik sem szignifikáns $
ightarrow$ **Klasszikus OLS**
   - Ha csak az egyik szignifikáns $
ightarrow$ **SAR** (Lag) vagy **SEM** (Error)
   - Ha mindkettő szignifikáns $
ightarrow$ A magasabb robusztus tesztstatisztikájú ($\max(	ext{RLM-Lag}, 	ext{RLM-Error})$) modell választandó.


In [ ]:
# ==============================================================================
# MODUL 5: ANSELIN-FÉLE LM DIAGNOSZTIKAI DÖNTÉSI FA
# ==============================================================================
def evaluate_anselin_lm_tree(ols_dict: dict, city_label: str) -> pd.DataFrame:
    rows = []
    alpha = 0.05
    for m_code, m in ols_dict.items():
        lm_lag_stat, lm_lag_p = m.lm_lag
        lm_err_stat, lm_err_p = m.lm_error
        rlm_lag_stat, rlm_lag_p = m.rlm_lag
        rlm_err_stat, rlm_err_p = m.rlm_error

        lag_sig = lm_lag_p < alpha
        err_sig = lm_err_p < alpha

        if not lag_sig and not err_sig:
            choice = "Klasszikus OLS (Nincs térbeli hatás)"
        elif lag_sig and not err_sig:
            choice = "SAR (Spatial Lag / Késleltetett modell)"
        elif not lag_sig and err_sig:
            choice = "SEM (Spatial Error / Hibamodell)"
        else:
            if rlm_lag_stat > rlm_err_stat:
                choice = "SAR (RLM-Lag domináns)"
            else:
                choice = "SEM (RLM-Error domináns)"

        rows.append({
            'Város': city_label,
            'Modell': m_code,
            'Specifikáció': SPATIAL_SPECS[m_code]['name'],
            'LM-Lag Stat': round(float(lm_lag_stat), 2),
            'LM-Lag p': round(float(lm_lag_p), 5),
            'LM-Error Stat': round(float(lm_err_stat), 2),
            'LM-Error p': round(float(lm_err_p), 5),
            'Robusztus LM-Lag Stat': round(float(rlm_lag_stat), 2),
            'Robusztus LM-Lag p': round(float(rlm_lag_p), 5),
            'Robusztus LM-Error Stat': round(float(rlm_err_stat), 2),
            'Robusztus LM-Error p': round(float(rlm_err_p), 5),
            'Anselin Döntés': choice
        })
    return pd.DataFrame(rows)

df_anselin_k = evaluate_anselin_lm_tree(ols_k, "Budapest X. Kőbánya")
df_anselin_l = evaluate_anselin_lm_tree(ols_l, "Bécs II. Leopoldstadt")
df_anselin_summary = pd.concat([df_anselin_k, df_anselin_l], ignore_index=True)

export_table(df_anselin_summary, "15_anselin_lm_dontesi_fa", "Anselin-féle Lagrange Multiplier (LM) diagnosztikai döntési fa")


## 6. Modul: Térbeli Késleltetett Modellek (SAR / Spatial Lag) Maximum Likelihood Becslése

### Matematikai specifikáció:
$$\mathbf{y} = \rho \mathbf{W}\mathbf{y} + \mathbf{X}\boldsymbol{\beta} + \boldsymbol{\varepsilon}, \quad \boldsymbol{\varepsilon} \sim \mathcal{N}(0, \sigma^2 \mathbf{I}_n)$$
- $\rho$ (autoregresszív térbeli paraméter): Megméri, hogy a szomszédos lakások árainak 1%-os változása hány százalékos árváltozást idéz elő a vizsgált ingatlanban.
- Becslési eljárás: Ord (1975) koncentrált Maximum Likelihood (ML) eljárás a $\ln |\mathbf{I}_n - \rho \mathbf{W}|$ Jacobi-determináns pontos kiszámításával.


In [ ]:
# ==============================================================================
# MODUL 6: SAR MODELLCSALÁD MAXIMUM LIKELIHOOD BECSLÉSE (k-NN 5 BÁZIS)
# ==============================================================================
def estimate_sar_family(gdf: gpd.GeoDataFrame, w_obj, city_label: str):
    y = gdf[['ln_fajlagos_ar']].values
    sar_dict = {}
    rows = []

    for m_code, spec in SPATIAL_SPECS.items():
        feat = spec['features'].copy()
        if m_code == 'M2' and 'd_vasutallomas_0_375' in feat and 'd_vasutallomas_375_750' in feat:
            if 'd_vasutallomas_0_375' in gdf.columns and gdf['d_vasutallomas_0_375'].sum() < 5 and 'd_vasutallomas_0_750' in gdf.columns:
                feat = [f for f in feat if f not in ['d_vasutallomas_0_375', 'd_vasutallomas_375_750']]
                feat.insert(feat.index('d_vasutallomas_750_1125'), 'd_vasutallomas_0_750')
        feat = [c for c in feat if c in gdf.columns and gdf[c].nunique() > 1]
        X = gdf[feat].values

        ml_lag = spreg.ML_Lag(y, X, w=w_obj, name_x=feat, name_y='ln_fajlagos_ar', name_ds=city_label)
        sar_dict[m_code] = ml_lag

        rho_val = float(ml_lag.rho)
        rho_se = float(ml_lag.vm[-1, -1]**0.5)
        rho_z, rho_p = ml_lag.z_stat[-1]

        rows.append({
            'Város': city_label,
            'Modell': m_code,
            'Specifikáció': spec['name'],
            'Térbeli Késleltetés Rho (W_y)': round(rho_val, 4),
            'Rho SE': round(float(rho_se), 4),
            'Rho z-érték': round(float(rho_z), 3),
            'Rho p-érték': round(float(rho_p), 5),
            'Pseudo R²': round(float(ml_lag.pr2), 4),
            'Log-Likelihood': round(float(ml_lag.logll), 2),
            'AIC': round(float(ml_lag.aic), 2),
            'Schwarz BIC': round(float(ml_lag.schwarz), 2)
        })

    return pd.DataFrame(rows), sar_dict

df_sar_metrics_k, sar_k = estimate_sar_family(gdf_k_pontos, w_dict_k['w_knn5'], "Budapest X. Kőbánya")
df_sar_metrics_l, sar_l = estimate_sar_family(gdf_l_pontos, w_dict_l['w_knn5'], "Bécs II. Leopoldstadt")
df_sar_summary = pd.concat([df_sar_metrics_k, df_sar_metrics_l], ignore_index=True)

export_table(df_sar_summary, "16_sar_modellek_illeszkedes", "SAR (Spatial Lag) Maximum Likelihood modellek illeszkedési statisztikái")

## 7. Modul: Térbeli Hibamodellek (SEM / Spatial Error) Maximum Likelihood Becslése

### Matematikai specifikáció:
$$\mathbf{y} = \mathbf{X}\boldsymbol{\beta} + \mathbf{u}, \quad \mathbf{u} = \lambda \mathbf{W}\mathbf{u} + \boldsymbol{\varepsilon}, \quad \boldsymbol{\varepsilon} \sim \mathcal{N}(0, \sigma^2 \mathbf{I}_n)$$
- $\lambda$ (térbeli hibaautokorrelációs paraméter): Megszűri azokat a nem megfigyelt helyi környezeti, zaj- vagy presztízsjellemzőket, amelyek térben összefüggenek a szomszédságban.
- Becslési eljárás: Maximum Likelihood (ML) a térbeli kovarianciamátrix inverziójával.


In [ ]:
# ==============================================================================
# MODUL 7: SEM MODELLCSALÁD MAXIMUM LIKELIHOOD BECSLÉSE (k-NN 5 BÁZIS)
# ==============================================================================
def estimate_sem_family(gdf: gpd.GeoDataFrame, w_obj, city_label: str):
    y = gdf[['ln_fajlagos_ar']].values
    sem_dict = {}
    rows = []

    for m_code, spec in SPATIAL_SPECS.items():
        feat = spec['features'].copy()
        if m_code == 'M2' and 'd_vasutallomas_0_375' in feat and 'd_vasutallomas_375_750' in feat:
            if 'd_vasutallomas_0_375' in gdf.columns and gdf['d_vasutallomas_0_375'].sum() < 5 and 'd_vasutallomas_0_750' in gdf.columns:
                feat = [f for f in feat if f not in ['d_vasutallomas_0_375', 'd_vasutallomas_375_750']]
                feat.insert(feat.index('d_vasutallomas_750_1125'), 'd_vasutallomas_0_750')
        feat = [c for c in feat if c in gdf.columns and gdf[c].nunique() > 1]
        X = gdf[feat].values

        ml_err = spreg.ML_Error(y, X, w=w_obj, name_x=feat, name_y='ln_fajlagos_ar', name_ds=city_label)
        sem_dict[m_code] = ml_err

        lam_val = float(ml_err.lam)
        lam_se = float(ml_err.vm[-1, -1]**0.5)
        lam_z, lam_p = ml_err.z_stat[-1]

        rows.append({
            'Város': city_label,
            'Modell': m_code,
            'Specifikáció': spec['name'],
            'Térbeli Hiba Lambda (W_u)': round(lam_val, 4),
            'Lambda SE': round(float(lam_se), 4),
            'Lambda z-érték': round(float(lam_z), 3),
            'Lambda p-érték': round(float(lam_p), 5),
            'Pseudo R²': round(float(ml_err.pr2), 4),
            'Log-Likelihood': round(float(ml_err.logll), 2),
            'AIC': round(float(ml_err.aic), 2),
            'Schwarz BIC': round(float(ml_err.schwarz), 2)
        })

    return pd.DataFrame(rows), sem_dict

df_sem_metrics_k, sem_k = estimate_sem_family(gdf_k_pontos, w_dict_k['w_knn5'], "Budapest X. Kőbánya")
df_sem_metrics_l, sem_l = estimate_sem_family(gdf_l_pontos, w_dict_l['w_knn5'], "Bécs II. Leopoldstadt")
df_sem_summary = pd.concat([df_sem_metrics_k, df_sem_metrics_l], ignore_index=True)

export_table(df_sem_summary, "17_sem_modellek_illeszkedes", "SEM (Spatial Error) Maximum Likelihood modellek illeszkedési statisztikái")

## 8. Modul: LeSage & Pace (2009) Térbeli Hatásfelbontás (Impact Decomposition)

### Matematikai alapok és definíciók:
A SAR modellben az endogén $\mathbf{W}\mathbf{y}$ tag miatt az OLS $\beta_r$ koefficiensek nem értelmezhetők közvetlen parciális deriváltként.
A parciális deriváltak mátrixa a $r$-edik magyarázó változóra:
$$\mathbf{S}_r(\mathbf{W}) = \frac{\partial \mathbf{y}}{\partial \mathbf{x}_r} = (\mathbf{I}_n - \rho \mathbf{W})^{-1} \beta_r$$

A LeSage & Pace (2009) szerinti skalár összesítések:
- **Közvetlen Hatás (Average Direct Impact):**
  $$\text{Direct}_r = \frac{1}{n} \text{tr}(\mathbf{S}_r(\mathbf{W}))$$
  (A saját ingatlan jellemzőjének megváltozása mekkora saját árváltozást eredményez, figyelembe véve a visszacsatolási hurkokat).
- **Teljes Hatás (Average Total Impact):**
  $$\text{Total}_r = \frac{1}{n} \mathbf{1}_n' \mathbf{S}_r(\mathbf{W}) \mathbf{1}_n = \frac{\beta_r}{1 - \rho} \quad (\text{sorstandardizált } \mathbf{W} \text{ esetén})$$
- **Közvetett Hatás (Average Indirect Impact / Spillover):**
  $$\text{Indirect}_r = \text{Total}_r - \text{Direct}_r$$
  (A szomszédos ingatlanok jellemzőinek aggregált tovagyűrűző hatása a vizsgált ingatlan árára).

A számítás mind a négy modellre (M1, M2, M3, M4) elkészül mindkét városra, a konstans helyes leválasztásával.

In [ ]:
# ==============================================================================
# MODUL 8: LESAGE & PACE TÉRBELI HATÁSFELBONTÁS SZÁMÍTÁSA (M1 - M4)
# ==============================================================================
def compute_lesage_pace_table(sar_model, w_obj, city_label: str, model_code: str) -> pd.DataFrame:
    n = sar_model.n
    W_dense = w_obj.full()[0]
    I_mat = np.eye(n)

    betas = sar_model.betas.flatten()
    rho = float(betas[-1]) # Utolsó paraméter a W_y
    beta_vars = betas[1:-1] # Változók koefficiensei (Konstans kihagyásával)
    var_names = sar_model.name_x[1:-1] # Változónevek (Konstans és W_y nélkül)

    # Invertált szorzómátrix (I - rho * W)^(-1)
    inv_mult = np.linalg.inv(I_mat - rho * W_dense)

    rows = []
    for i, vname in enumerate(var_names):
        b = beta_vars[i]
        S_r = inv_mult * b

        direct = float(np.trace(S_r) / n)
        total = float(b / (1.0 - rho))
        indirect = total - direct

        rows.append({
            'Város': city_label,
            'Modell': model_code,
            'Specifikáció': SPATIAL_SPECS[model_code]['name'],
            'Változó': vname,
            'Nyers Beta': round(float(b), 4),
            'Közvetlen Hatás (Direct)': round(direct, 4),
            'Közvetett Hatás (Indirect)': round(indirect, 4),
            'Teljes Hatás (Total)': round(total, 4),
            'Közvetett Hatás Arány (%)': round(float(indirect / total * 100), 1) if abs(total) > 1e-6 else 0.0
        })
    return pd.DataFrame(rows)

# Hatásfelbontás kiszámítása mind a 4 modellre mindkét városban
df_impacts_k = pd.concat([compute_lesage_pace_table(sar_k[m], w_dict_k['w_knn5'], "Budapest X. Kőbánya", m) for m in ['M1', 'M2', 'M3', 'M4']], ignore_index=True)
df_impacts_l = pd.concat([compute_lesage_pace_table(sar_l[m], w_dict_l['w_knn5'], "Bécs II. Leopoldstadt", m) for m in ['M1', 'M2', 'M3', 'M4']], ignore_index=True)
df_impacts_summary = pd.concat([df_impacts_k, df_impacts_l], ignore_index=True)

export_table(df_impacts_summary, "18_lesage_pace_hatasfelbontas", "LeSage & Pace (2009) térbeli közvetlen, közvetett és teljes hatásfelbontás (M1 - M4)")

## 9. Modul: Átfogó Modell-összehasonlítás (OLS vs. SAR vs. SEM)

### Specifikációs értékelési metrikák:
Összevetjük a modellek illeszkedését a három fő ökonometriai keretrendszerben (OLS, SAR, SEM) az információs kritériumok ($	ext{AIC}, 	ext{BIC}$) és a magyarázóerő ($R^2 / 	ext{Pseudo } R^2$) alapján.


In [ ]:
# ==============================================================================
# MODUL 9: ÁTFOGÓ ÖKONOMETRIAI NAGYTÁBLÁZAT (OLS VS. SAR VS. SEM)
# ==============================================================================
def build_grand_comparison(city_label: str, ols_dict: dict, sar_dict: dict, sem_dict: dict) -> pd.DataFrame:
    rows = []
    for m_code in ['M1', 'M2', 'M3', 'M4']:
        m_ols = ols_dict[m_code]
        m_sar = sar_dict[m_code]
        m_sem = sem_dict[m_code]

        rows.append({
            'Város': city_label,
            'Modell': m_code,
            'Specifikáció': SPATIAL_SPECS[m_code]['name'],
            'OLS R²': round(float(m_ols.r2), 4),
            'OLS AIC': round(float(m_ols.aic), 2),
            'OLS BIC': round(float(m_ols.schwarz), 2),
            'SAR Rho (Wy)': round(float(m_sar.rho), 4),
            'SAR Pseudo R²': round(float(m_sar.pr2), 4),
            'SAR AIC': round(float(m_sar.aic), 2),
            'SAR BIC': round(float(m_sar.schwarz), 2),
            'SEM Lambda (Wu)': round(float(m_sem.lam), 4),
            'SEM Pseudo R²': round(float(m_sem.pr2), 4),
            'SEM AIC': round(float(m_sem.aic), 2),
            'SEM BIC': round(float(m_sem.schwarz), 2)
        })
    return pd.DataFrame(rows)

df_grand_k = build_grand_comparison("Budapest X. Kőbánya", ols_k, sar_k, sem_k)
df_grand_l = build_grand_comparison("Bécs II. Leopoldstadt", ols_l, sar_l, sem_l)
df_grand_summary = pd.concat([df_grand_k, df_grand_l], ignore_index=True)

export_table(df_grand_summary, "19_grand_spatial_comparison", "Átfogó ökonometriai modell-összehasonlítás (OLS vs. SAR vs. SEM)")


## 10. Modul: Térbeli Súlymátrixok Robusztussági Elemzése ($k$-NN 5 vs. $k$-NN 8 vs. Inverz Távolság)

### Módszertani megalapozás:
A térbeli ökonometriai becslések érzékenységének vizsgálatára mind a négy modellspeficikációt (M1–M4) újra megbecsüljük a három eltérő súlymátrix-topológiával:
1. **$k$-NN 5:** Elsődleges bázismátrix (5 legközelebbi szomszéd).
2. **$k$-NN 8:** Tágabb lokális szomszédság (8 legközelebbi szomszéd).
3. **Inverz Távolság ($1/d$):** Folyamatos távolság alapú lecsengés 2500 méteres küszöbön belül.

A robusztussági nagytáblázat bemutatja a térbeli késleltetési ($ho$) és hibaparaméterek ($\lambda$) stabilitását, szignifikanciáját, valamint a modellek relatív illeszkedését (AIC, Log-Likelihood).

In [ ]:
# ==============================================================================
# MODUL 10: SÚLYMÁTRIX ROBUSZTUSSÁGI NAGYTÁBLÁZAT (k-NN 5 vs. k-NN 8 vs. 1/d)
# ==============================================================================
def compute_spatial_weights_robustness(gdf: gpd.GeoDataFrame, w_dict: dict, city_label: str) -> pd.DataFrame:
    y = gdf[['ln_fajlagos_ar']].values
    rows = []
    matrix_labels = [
        ('w_knn5', 'k-NN 5 (Bázis)'),
        ('w_knn8', 'k-NN 8'),
        ('w_inv', 'Inverz távolság (1/d, 2500m)')
    ]

    for m_code, spec in SPATIAL_SPECS.items():
        feat = [c for c in spec['features'] if c in gdf.columns and gdf[c].nunique() > 1]
        X = gdf[feat].values

        for w_key, w_name in matrix_labels:
            w_obj = w_dict[w_key]

            # SAR becslés
            sar_m = spreg.ML_Lag(y, X, w=w_obj, name_x=feat, name_y='ln_fajlagos_ar')
            rho_val = float(sar_m.rho)
            rho_se = float(sar_m.vm[-1, -1]**0.5)
            rho_z, rho_p = sar_m.z_stat[-1]

            # SEM becslés
            sem_m = spreg.ML_Error(y, X, w=w_obj, name_x=feat, name_y='ln_fajlagos_ar')
            lam_val = float(sem_m.lam)
            lam_se = float(sem_m.vm[-1, -1]**0.5)
            lam_z, lam_p = sem_m.z_stat[-1]

            rows.append({
                'Város': city_label,
                'Modell': m_code,
                'Specifikáció': spec['name'],
                'Súlymátrix': w_name,
                'SAR Rho (Wy)': round(rho_val, 4),
                'SAR Rho SE': round(float(rho_se), 4),
                'SAR Rho p': round(float(rho_p), 5),
                'SAR Pseudo R²': round(float(sar_m.pr2), 4),
                'SAR AIC': round(float(sar_m.aic), 2),
                'SEM Lambda (Wu)': round(lam_val, 4),
                'SEM Lambda SE': round(float(lam_se), 4),
                'SEM Lambda p': round(float(lam_p), 5),
                'SEM Pseudo R²': round(float(sem_m.pr2), 4),
                'SEM AIC': round(float(sem_m.aic), 2)
            })

    return pd.DataFrame(rows)

df_spat_rob_k = compute_spatial_weights_robustness(gdf_k_pontos, w_dict_k, "Budapest X. Kőbánya")
df_spat_rob_l = compute_spatial_weights_robustness(gdf_l_pontos, w_dict_l, "Bécs II. Leopoldstadt")
df_spat_rob_summary = pd.concat([df_spat_rob_k, df_spat_rob_l], ignore_index=True)

export_table(df_spat_rob_summary, "19b_terbeli_sulymatrix_robusztussag", "Térbeli súlymátrixok robusztussági összehasonlítása (k-NN 5 vs. k-NN 8 vs. Inverz távolság)")

# ==============================================================================
# 4. FŐRÉSZ: FÖLDRAJZI SÚLYOZÁSÚ REGRESSZIÓ (GWR & MGWR)
# ==============================================================================


## 2. Modul: Adatbetöltés és Hivatalos Síkvetületi Transzformáció

### Matematikai és vetületi specifikáció:
A GWR kerneltávolságainak $d_{ij} = \sqrt{(u_i - u_j)^2 + (v_i - v_j)^2}$ metrikus pontosságához a koordinátákat hivatalos vetületbe transzformáljuk:
- **Kőbánya:** EPSG:23700 (HD72 / EOV).
- **Leopoldstadt:** EPSG:31256 (MGI / Austria GK East).


In [ ]:
# ==============================================================================
# MODUL 2: GWR SÍKVETÜLETI ADATOK CSATOLÁSA
# ==============================================================================
# A 3. Főrésznél már létrehozott vetületi GeoDataFrame-ek újrahasznosítása a maximális koherencia érdekében:
gdf_gwr_k = gdf_k_pontos.copy()
gdf_gwr_l = gdf_l_pontos.copy()
log_info(f"GWR térinformatikai rétegek csatolva: Kőbánya (N={len(gdf_gwr_k)}), Leopoldstadt (N={len(gdf_gwr_l)})")


## 3. Modul: Változók Z-score Standardizálása

### Matematikai specifikáció:
A különböző mértékegységek ($m^2$, év, méter) skálahatásának semlegesítésére minden magyarázó és függő változót Z-score transzformációnak vetünk alá:
$$z_{ik} = \frac{x_{ik} - \bar{x}_k}{s_k}, \quad z_{iy} = \frac{y_i - \bar{y}}{s_y}$$
Ennek révén a becsült lokális $\beta_k(u_i, v_i)$ koefficiensek standardizált bétaként (Beta weights) értelmezhetők.


In [ ]:
# ==============================================================================
# MODUL 3: Z-SCORE STANDARDIZÁLÁS ÉS ADATMÁTRIXOK ELŐKÉSZÍTÉSE
# ==============================================================================
GWR_FEATURES = [
    'alapterulet_nm', 'szobak_szama', 'd_felujitott',
    'ln_dist_vagany_m', 'ln_dist_vasutallomas_hm', 'ln_dist_fout_m'
]

FEATURE_LABELS_GWR = {
    'Konstans (Intercept)': 'Konstans (Intercept)',
    'Konstans': 'Konstans (Intercept)',
    'alapterulet_nm': 'Alapterület (m²)',
    'szobak_szama': 'Szobák száma (db)',
    'd_felujitott': 'Felújított állapot dummy',
    'ln_dist_vagany_m': 'ln(Vasúti vágány távolság, m)',
    'ln_dist_vasutallomas_hm': 'ln(Vasútállomás távolság, m)',
    'ln_dist_fout_m': 'ln(Főútvonal távolság, m)'
}

def prepare_gwr_arrays(gdf: gpd.GeoDataFrame):
    valid_cols = [c for c in GWR_FEATURES if c in gdf.columns]
    df_clean = gdf.dropna(subset=valid_cols + ['ln_fajlagos_ar', 'u_proj', 'v_proj']).copy()

    # Koordináták mikrojitterrel (1.0m szórás) a szinguláris kernelmátrixok megelőzésére
    rng = np.random.default_rng(42)
    coords_raw = np.array(list(zip(df_clean['u_proj'], df_clean['v_proj'])))
    coords = coords_raw + rng.normal(0, 1.0, coords_raw.shape)

    y_raw = df_clean['ln_fajlagos_ar'].values.reshape(-1, 1)
    y_std = (y_raw - np.mean(y_raw)) / np.std(y_raw)

    X_raw = df_clean[valid_cols].values
    X_std = (X_raw - np.mean(X_raw, axis=0)) / np.std(X_raw, axis=0)

    return {
        'coords': coords,
        'y_std': y_std,
        'X_std': X_std,
        'var_names': valid_cols,
        'var_labels': [FEATURE_LABELS_GWR.get(c, c) for c in valid_cols],
        'N': len(df_clean)
    }

data_gwr_k = prepare_gwr_arrays(gdf_gwr_k)
data_gwr_l = prepare_gwr_arrays(gdf_gwr_l)
log_info("GWR numerikus tömbök és koordináták sikeresen Z-score standardizálva (M3 log-távolságokkal).")

## 4. Modul: Optimális Adaptív Sávszélesség Keresése (Bandwidth Selection)

### Optimalizálási algoritmus és kritérium:
A térbeli súlyozáshoz adaptív bisquare kernelfüggvényt alkalmazunk:
$$w_{ij} = \begin{cases} \left(1 - \left(\frac{d_{ij}}{b_i}\right)^2\right)^2, & \text{ha } d_{ij} < b_i \\ 0, & \text{egyébként} \end{cases}$$
ahol $b_i$ az $i$-edik pont $k$-adik legközelebbi szomszédjának távolsága (adaptív sávszélesség).

Az optimális szomszédszám keresése aranymetszéses kereséssel (Golden Section Search) történik a korrigált Akaike Információs Kritérium ($	ext{AICc}$) minimalizálásával:
$$\text{AICc} = 2n \ln(\hat{\sigma}) + n \ln(2\pi) + n \left( \frac{n + \operatorname{tr}(\mathbf{S})}{n - 2 - \operatorname{tr}(\mathbf{S})} \right)$$


In [ ]:
# ==============================================================================
# MODUL 4: OPTIMÁLIS SÁVSZÉLESSÉG KERESÉS
# ==============================================================================
def find_optimal_bandwidth_grid(data_dict: dict, city_label: str) -> float:
    coords = data_dict['coords']
    y = data_dict['y_std']
    X = data_dict['X_std']
    n = data_dict['N']

    candidates = [int(n * frac) for frac in [0.35, 0.40, 0.45, 0.50, 0.55, 0.60, 0.65, 0.70, 0.80]]
    best_bw = None
    best_aicc = float('inf')

    for bw in candidates:
        try:
            model = GWR(coords, y, X, bw=bw, kernel='bisquare', fixed=False, constant=True, n_jobs=1)
            res = model.fit()
            if res.aicc < best_aicc:
                best_aicc = res.aicc
                best_bw = bw
        except Exception:
            continue

    if best_bw is None:
        best_bw = int(n * 0.60)

    log_info(f"{city_label}: Optimális adaptív sávszélesség rácskereséssel: b = {best_bw} szomszéd (AICc={best_aicc:.2f}).")
    return float(best_bw)

opt_bw_k = find_optimal_bandwidth_grid(data_gwr_k, "Budapest X. Kőbánya")
opt_bw_l = find_optimal_bandwidth_grid(data_gwr_l, "Bécs II. Leopoldstadt")

df_bw = pd.DataFrame([
    {
        'Város': "Budapest X. Kőbánya",
        'Minta N': data_gwr_k['N'],
        'Optimális Sávszélesség (szomszédszám b)': int(opt_bw_k),
        'Minta arány (%)': round(opt_bw_k / data_gwr_k['N'] * 100, 1),
        'Kernel': 'Adaptív Bisquare',
        'Kritérium': 'AICc minimalizálás'
    },
    {
        'Város': "Bécs II. Leopoldstadt",
        'Minta N': data_gwr_l['N'],
        'Optimális Sávszélesség (szomszédszám b)': int(opt_bw_l),
        'Minta arány (%)': round(opt_bw_l / data_gwr_l['N'] * 100, 1),
        'Kernel': 'Adaptív Bisquare',
        'Kritérium': 'AICc minimalizálás'
    }
])
export_table(df_bw, "20_gwr_optimalis_savszelesseg", "Optimális adaptív GWR sávszélességek (AICc)")


## 5. Modul: GWR Modellbecslés és Globális OLS vs. GWR Illeszkedés

### Modell-összehasonlítási metrikák:
Összevetjük a globális OLS modellt és a lokális GWR specifikációt:
- Reziduális négyzetösszeg ($RSS$)
- Hatásos paraméterszám ($\operatorname{tr}(\mathbf{S})$)
- Korrigált $R^2$ és $	ext{AICc}$ növekmény / csökkenés.


In [ ]:
# ==============================================================================
# MODUL 5: GWR ILLESZTÉS ÉS GLOBÁLIS OLS VS. LOKÁLIS GWR ÖSSZEVETÉS
# ==============================================================================
def fit_and_evaluate_gwr(data_dict: dict, opt_bw: float, city_label: str):
    coords = data_dict['coords']
    y = data_dict['y_std']
    X = data_dict['X_std']

    # Globális OLS referencia számítása összevetéshez
    X_with_const = sm.add_constant(X)
    ols_global = sm.OLS(y, X_with_const).fit()
    ols_r2 = float(ols_global.rsquared)
    ols_aic = float(ols_global.aic)

    gwr_model = GWR(coords, y, X, bw=opt_bw, kernel='bisquare', fixed=False, constant=True, n_jobs=1)
    gwr_res = gwr_model.fit()

    metrics = {
        'Város': city_label,
        'Sávszélesség (b)': int(opt_bw),
        'Globális OLS R²': round(ols_r2, 4),
        'Globális OLS AIC': round(ols_aic, 2),
        'Lokális GWR R²': round(float(gwr_res.R2), 4),
        'Lokális GWR Adj-R²': round(float(gwr_res.adj_R2), 4),
        'GWR AIC': round(float(gwr_res.aic), 2),
        'GWR AICc': round(float(gwr_res.aicc), 2),
        'GWR BIC': round(float(gwr_res.bic), 2),
        'Hatásos szabadságfok (tr(S))': round(float(gwr_res.tr_S), 2)
    }
    return gwr_res, metrics

gwr_res_k, gwr_metrics_k = fit_and_evaluate_gwr(data_gwr_k, opt_bw_k, "Budapest X. Kőbánya")
gwr_res_l, gwr_metrics_l = fit_and_evaluate_gwr(data_gwr_l, opt_bw_l, "Bécs II. Leopoldstadt")

df_gwr_comparison = pd.DataFrame([gwr_metrics_k, gwr_metrics_l])
export_table(df_gwr_comparison, "21_gwr_modell_illeszkedes", "Globális OLS vs. Lokális GWR illeszkedési statisztikák")

## 6. Modul: Lokális Paraméterek Statisztikai Eloszlása (5-Számos Összefoglaló)

### Módszertani specifikáció:
A GWR minden megfigyelési pontra egyedi $\hat{eta}_k(u_i, v_i)$ értéket és lokális $t$-statisztikát számít. A paraméterek térbeli változékonyságát 5-számos statisztikai összefoglalóval (Min, $Q_1$, Medián, $Q_3$, Max) és a szignifikáns megfigyelések arányával ($|t| > 1.96, p < 0.05$) számszerűsítjük.


In [ ]:
# ==============================================================================
# MODUL 6: LOKÁLIS PARAMÉTEREK 5-SZÁMOS ELOSZLÁSÁNAK SZÁMÍTÁSA
# ==============================================================================
def summarize_gwr_parameters(gwr_res, var_names: list, city_label: str) -> pd.DataFrame:
    all_names = ['Konstans (Intercept)'] + var_names
    params = gwr_res.params
    tvals = gwr_res.tvalues

    rows = []
    for idx, name in enumerate(all_names):
        p_vec = params[:, idx]
        t_vec = tvals[:, idx]
        sig_pct = (np.abs(t_vec) > 1.96).mean() * 100.0

        rows.append({
            'Város': city_label,
            'Változó': name,
            'Változó Címke': FEATURE_LABELS_GWR.get(name, name),
            'Min': round(float(np.min(p_vec)), 4),
            'Alsó Kvartilis (Q1)': round(float(np.percentile(p_vec, 25)), 4),
            'Medián (Q2)': round(float(np.median(p_vec)), 4),
            'Felső Kvartilis (Q3)': round(float(np.percentile(p_vec, 75)), 4),
            'Max': round(float(np.max(p_vec)), 4),
            'IQR Terjedelem': round(float(np.percentile(p_vec, 75) - np.percentile(p_vec, 25)), 4),
            'Szignifikáns pontok aránya (%)': round(float(sig_pct), 1)
        })
    return pd.DataFrame(rows)

df_params_gwr_k = summarize_gwr_parameters(gwr_res_k, data_gwr_k['var_names'], "Budapest X. Kőbánya")
df_params_gwr_l = summarize_gwr_parameters(gwr_res_l, data_gwr_l['var_names'], "Bécs II. Leopoldstadt")
df_gwr_params_summary = pd.concat([df_params_gwr_k, df_params_gwr_l], ignore_index=True)

export_table(df_gwr_params_summary, "22_gwr_lokalis_parameterek_eloszlasa", "GWR lokális paraméterek 5-számos eloszlása és szignifikanciája")


## 7. Modul: Multiscale GWR (MGWR) – Változónként Eltérő Térbeli Léptékek

### Matematikai specifikáció:
A Multiscale GWR (Fotheringham et al., 2017) feloldja a klasszikus GWR azon korlátját, hogy minden változóra azonos térbeli sávszélességet alkalmaz. Az MGWR egyenlete:
$$y_i = eta_0(u_i, v_i) + \sum_{k=1}^p eta_{bw_k}(u_i, v_i) x_{ik} + arepsilon_i$$
ahol minden $k$-adik magyarázó változó saját, optimalizált $bw_k$ sávszélességgel rendelkezik.


In [ ]:
# ==============================================================================
# MODUL 7: MULTISCALE GWR (MGWR) BECSLÉS (VÉDETT BLOKK)
# ==============================================================================
def estimate_mgwr_summary(data_dict: dict, city_label: str) -> pd.DataFrame:
    coords = data_dict['coords']
    y = data_dict['y_std']
    X = data_dict['X_std']
    var_names = ['Konstans'] + data_dict['var_names']

    log_info(f"{city_label}: MGWR változónkénti sávszélesség-keresés indítása...")
    try:
        mgwr_selector = Sel_BW(coords, y, X, multi=True, kernel='bisquare', fixed=False, constant=True, n_jobs=1)
        mgwr_bw = mgwr_selector.search(max_iter_multi=15, criterion='AICc')

        mgwr_model = MGWR(coords, y, X, selector=mgwr_selector, kernel='bisquare', fixed=False, constant=True, n_jobs=1)
        mgwr_res = mgwr_model.fit()

        rows = []
        for idx, name in enumerate(var_names):
            bw_val = float(mgwr_bw[idx]) if hasattr(mgwr_bw, '__getitem__') else float(mgwr_bw)
            scale_type = 'Globális lépték' if bw_val > len(coords) * 0.75 else ('Helyi / Lokális lépték' if bw_val < len(coords) * 0.3 else 'Köztes / Regionális lépték')
            rows.append({
                'Város': city_label,
                'Változó': name,
                'Változó Címke': FEATURE_LABELS_GWR.get(name, name),
                'MGWR Sávszélesség (szomszédszám)': int(bw_val),
                'Minta arány (%)': round(bw_val / len(coords) * 100, 1),
                'Térbeli Működési Lépték': scale_type
            })
        return pd.DataFrame(rows)
    except Exception as e:
        log_info(f"{city_label}: MGWR számítás korlátozottsága kis mintán ({e}). A standard GWR eredmények tekintendők irányadónak.")
        rows = []
        for name in var_names:
            rows.append({
                'Város': city_label,
                'Változó': name,
                'Változó Címke': FEATURE_LABELS_GWR.get(name, name),
                'MGWR Sávszélesség (szomszédszám)': np.nan,
                'Minta arány (%)': np.nan,
                'Térbeli Működési Lépték': f'Lokális szingularitás kis mintán ({str(e)[:30]})'
            })
        return pd.DataFrame(rows)

df_mgwr_list = []
for label, d in [("Budapest X. Kőbánya", data_gwr_k), ("Bécs II. Leopoldstadt", data_gwr_l)]:
    df_res = estimate_mgwr_summary(d, label)
    if not df_res.empty:
        df_mgwr_list.append(df_res)

if df_mgwr_list:
    df_mgwr_summary = pd.concat(df_mgwr_list, ignore_index=True)
    export_table(df_mgwr_summary, "23_mgwr_valtozonkenti_savszelessegek", "Multiscale GWR (MGWR) változónkénti térbeli működési léptékek")

# ==============================================================================
# 5. FŐRÉSZ: GÉPI TANULÁSI BENCHMARK ÉS XAI ÉRTELMEZÉS
# ==============================================================================


## 2. Modul: Adatbetöltés és 80-20%-os Tanító-Teszt Bontás

### Módszertani specifikáció:
A túlilleszkedés (overfitting) megelőzése és a valós out-of-sample általánosító képesség mérése érdekében az adatbázist 80% tanító és 20% független teszt halmazra osztjuk fel rögzített véletlenszám-maggal (`random_state=42`).


In [ ]:
# ==============================================================================
# MODUL 2: GÉPI TANULÁSI ADATMÁTRIXOK ÉS 80-20%-OS TANÍTÓ-TESZT BONTÁS
# ==============================================================================
from sklearn.model_selection import train_test_split, KFold, cross_val_score
from sklearn.linear_model import LinearRegression, RidgeCV, ElasticNetCV
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
import xgboost as xgb
import lightgbm as lgb
try:
    import catboost as cb
except ImportError:
    cb = None
import shap

ML_FEATURES = [
    'alapterulet_nm', 'szobak_szama', 'd_felujitott', 'd_felujitando',
    'd_lift', 'd_kulteri_kapcsolat', 'kulteri_alapterulet_nm', 'd_panel',
    'd_szuteren', 'd_foldszint', 'd_zaroszint', 'd_epult_2015_utan',
    'ln_dist_fout_m', 'ln_dist_vagany_m', 'ln_dist_vasutallomas_hm'
]

def prepare_and_split_ml_data(df: pd.DataFrame, city_label: str) -> dict:
    valid_features = [c for c in ML_FEATURES if c in df.columns and df[c].nunique() > 1]
    df_clean = df.dropna(subset=valid_features + ['ln_fajlagos_ar']).copy()

    X = df_clean[valid_features]
    y = df_clean['ln_fajlagos_ar']

    from sklearn.model_selection import GroupShuffleSplit
    groups = df_clean['cim_teljes'].fillna('ismeretlen')
    gss = GroupShuffleSplit(n_splits=1, test_size=0.20, random_state=42)
    train_idx, test_idx = next(gss.split(X, y, groups=groups))

    X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
    y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]

    log_info(f"{city_label}: ML adatmátrix előállítva (Címcsoportosított GroupShuffleSplit 80-20 bontás, Tanító N={len(X_train)}, Teszt N={len(X_test)}, Változók K={len(valid_features)})")
    return {
        'X_train': X_train,
        'X_test': X_test,
        'y_train': y_train,
        'y_test': y_test,
        'features': valid_features,
        'city': city_label
    }

split_data_k = prepare_and_split_ml_data(df_k_pontos, "Budapest X. Kőbánya")
split_data_l = prepare_and_split_ml_data(df_l_pontos, "Bécs II. Leopoldstadt")

## 3. Modul: Algoritmuscsaládok Regisztere és 5-Fold Keresztvalidáció

### Matematikai specifikáció és metrikák:
A modellek becslése és összehasonlítása 5-fold keresztvalidációval és a független teszthalmazon történik:
- **Gyök-Átlagos Négyzetes Hiba (Root Mean Squared Error):**
  $$\text{RMSE} = \sqrt{\frac{1}{n} \sum_{i=1}^n (y_i - \hat{y}_i)^2}$$
- **Átlagos Abszolút Hiba (Mean Absolute Error):**
  $$\text{MAE} = \frac{1}{n} \sum_{i=1}^n |y_i - \hat{y}_i|$$
- **Determinációs Együttható ($R^2$):**
  $$R^2 = 1 - \frac{\sum_{i=1}^n (y_i - \hat{y}_i)^2}{\sum_{i=1}^n (y_i - \bar{y})^2}$$

Az algoritmusok és hiperparamétereik az `ML_MODEL_REGISTRY` szótárban közvetlenül módosíthatók.


In [ ]:
# ==============================================================================
# MODUL 3: ALGORITMUSOK REGISZTERE ÉS UNIVERZÁLIS BENCHMARK MOTOR (7 MODELL)
# ==============================================================================
def get_ml_registry():
    registry = {
        '1. OLS Baseline': LinearRegression(),
        '2. Ridge Regresszió': RidgeCV(alphas=np.logspace(-3, 3, 50)),
        '3. ElasticNet': ElasticNetCV(l1_ratio=[.1, .5, .7, .9, .95, .99, 1], cv=5, random_state=42),
        '4. Random Forest': RandomForestRegressor(n_estimators=300, max_depth=12, random_state=42, n_jobs=1),
        '5. XGBoost': xgb.XGBRegressor(n_estimators=300, max_depth=5, learning_rate=0.05, subsample=0.8, colsample_bytree=0.8, random_state=42),
        '6. LightGBM': lgb.LGBMRegressor(n_estimators=300, max_depth=5, learning_rate=0.05, subsample=0.8, random_state=42, verbose=-1)
    }
    if cb is not None:
        registry['7. CatBoost'] = cb.CatBoostRegressor(iterations=300, depth=5, learning_rate=0.05, random_seed=42, verbose=0)
    return registry

def run_ml_benchmark_suite(split_dict: dict, city_label: str):
    X_train = split_dict['X_train']
    y_train = split_dict['y_train']
    X_test = split_dict['X_test']
    y_test = split_dict['y_test']

    cv = KFold(n_splits=5, shuffle=True, random_state=42)
    rows = []
    trained_models = {}

    model_registry = get_ml_registry()
    for model_name, model_inst in model_registry.items():
        # 5-fold CV tanító halmazon
        cv_scores = cross_val_score(model_inst, X_train, y_train, cv=cv, scoring='neg_root_mean_squared_error')
        cv_rmse = -np.mean(cv_scores)

        # Illesztés és tesztpredikció
        model_inst.fit(X_train, y_train)
        y_pred = model_inst.predict(X_test)
        trained_models[model_name] = model_inst

        test_rmse = np.sqrt(mean_squared_error(y_test, y_pred))
        test_mae = mean_absolute_error(y_test, y_pred)
        test_r2 = r2_score(y_test, y_pred)

        # Eredeti árszinten vett százalékos hiba (MAPE)
        test_mape = np.mean(np.abs((np.exp(y_test) - np.exp(y_pred)) / np.exp(y_test))) * 100

        rows.append({
            'Város': city_label,
            'Modell': model_name,
            '5-Fold CV RMSE': round(float(cv_rmse), 4),
            'Teszt RMSE': round(float(test_rmse), 4),
            'Teszt MAE': round(float(test_mae), 4),
            'Teszt R²': round(float(test_r2), 4),
            'Teszt MAPE (%)': round(float(test_mape), 2)
        })

    df_res = pd.DataFrame(rows).sort_values('Teszt RMSE')
    log_info(f"{city_label}: Benchmark futtatás kész. Legjobb modell: {df_res.iloc[0]['Modell']} (Teszt RMSE: {df_res.iloc[0]['Teszt RMSE']:.4f})")
    return df_res, trained_models

df_bench_k, trained_k = run_ml_benchmark_suite(split_data_k, "Budapest X. Kőbánya")
df_bench_l, trained_l = run_ml_benchmark_suite(split_data_l, "Bécs II. Leopoldstadt")
df_benchmark_grand = pd.concat([df_bench_k, df_bench_l], ignore_index=True)

export_table(df_benchmark_grand, "24_ml_benchmark_rangsor", "Gépi tanulási algoritmusok out-of-sample predikciós rangsora")

## 4. Modul: SHAP Globális Változó-fontosság (TreeExplainer)

### Matematikai specifikáció:
A Shapley-érték (Lundberg & Lee 2017) kooperatív játékelméleti axiómákon (hatékonyság, szimmetria, additivitás, dummy) alapuló parciális hozzájárulás:
$$\phi_j = \sum_{S \subseteq F \setminus \{j\}} \frac{|S|!(|F| - |S| - 1)!}{|F|!} \left( f(S \cup \{j\}) - f(S) \right)$$
A globális változó-fontosságot az abszolút Shapley-értékek átlaga adja:
$$I_j = \frac{1}{n} \sum_{i=1}^n |\phi_{i, j}|$$


In [ ]:
# ==============================================================================
# MODUL 4: SHAP TREEEXPLAINER ÉS VÁLTOZÓ-FONTOSSÁG SZÁMÍTÁSA
# ==============================================================================
def compute_shap_importance(split_dict: dict, trained_dict: dict, df_bench: pd.DataFrame, city_label: str) -> pd.DataFrame:
    # Fa-alapú modellek közül a legkisebb Teszt RMSE-vel rendelkező kiválasztása
    tree_candidates = ['4. Random Forest', '5. XGBoost', '6. LightGBM', '7. CatBoost']
    df_tree = df_bench[(df_bench['Város'] == city_label) & (df_bench['Modell'].isin(tree_candidates))]
    best_tree_name = df_tree.iloc[0]['Modell'] if not df_tree.empty else '5. XGBoost'

    log_info(f"{city_label}: SHAP TreeExplainer illesztése a nyertes fa-modellre: {best_tree_name}")
    model = trained_dict[best_tree_name]
    X_train = split_dict['X_train']

    explainer = shap.TreeExplainer(model)
    shap_vals = explainer(X_train)

    mean_abs_shap = np.abs(shap_vals.values).mean(axis=0)
    features = list(X_train.columns)
    total_shap = np.sum(mean_abs_shap)

    rows = []
    for f, val in zip(features, mean_abs_shap):
        rel_weight = (val / total_shap) * 100 if total_shap > 0 else 0.0
        rows.append({
            'Város': city_label,
            'Nyertes Modell': best_tree_name,
            'Változó': f,
            'Változó Címke': VARIABLE_LABELS.get(f, f),
            'Átlagos |SHAP| érték': round(float(val), 4),
            'Relatív Súly (%)': round(float(rel_weight), 2)
        })

    df_shap = pd.DataFrame(rows).sort_values('Átlagos |SHAP| érték', ascending=False)
    df_shap['Fontossági Rangsor'] = range(1, len(df_shap) + 1)
    return df_shap

df_shap_k = compute_shap_importance(split_data_k, trained_k, df_bench_k, "Budapest X. Kőbánya")
df_shap_l = compute_shap_importance(split_data_l, trained_l, df_bench_l, "Bécs II. Leopoldstadt")
df_shap_summary = pd.concat([df_shap_k, df_shap_l], ignore_index=True)

export_table(df_shap_summary, "25_shap_globalis_valtozofontossag", "SHAP globális változó-fontosság rangsor és relatív súlyok")

## 5. Modul: Csoportosított Infrastrukturális és Ingatlanpiaci Hatásarányok

### Csoportosítási struktúra:
Az egyedi változók relatív SHAP súlyait 3 funkcionális csoportba aggregáljuk az infrastrukturális hatások számszerűsítéséhez:
1. **Fizikai lakásjellemzők:** Alapterület, szobaszám, állapot, kor, lift, erkély, panel, szuterén, szint.
2. **Vasúti infrastruktúra externáliák:** Vasúti sín zajtávolság és vasútállomás gyalogos elérhetőség.
3. **Közúti infrastruktúra externáliák:** Főútvonal távolság.


In [ ]:
# ==============================================================================
# MODUL 5: CSOPORTOSÍTOTT SHAP FONTOSSÁGOK SZÁMÍTÁSA
# ==============================================================================
def categorize_feature(var_name: str) -> str:
    if 'vagany' in var_name or 'vasutallomas' in var_name:
        return 'Vasúti infrastruktúra (Zaj + Elérhetőség)'
    elif 'fout' in var_name:
        return 'Közúti infrastruktúra (Főút zaj)'
    else:
        return 'Fizikai ingatlantulajdonságok'

def aggregate_grouped_importance(df_shap: pd.DataFrame, city_label: str) -> pd.DataFrame:
    df_sub = df_shap[df_shap['Város'] == city_label].copy()
    df_sub['Kategória'] = df_sub['Változó'].apply(categorize_feature)

    grp = df_sub.groupby('Kategória')['Relatív Súly (%)'].sum().reset_index()
    grp['Város'] = city_label
    grp['Aggregált Súly (%)'] = grp['Relatív Súly (%)'].round(2)
    return grp[['Város', 'Kategória', 'Aggregált Súly (%)']].sort_values('Aggregált Súly (%)', ascending=False)

df_grp_k = aggregate_grouped_importance(df_shap_k, "Budapest X. Kőbánya")
df_grp_l = aggregate_grouped_importance(df_shap_l, "Bécs II. Leopoldstadt")
df_grouped_importance = pd.concat([df_grp_k, df_grp_l], ignore_index=True)

export_table(df_grouped_importance, "26_csoportositott_fontossag_osszehasonlitas", "Csoportosított infrastrukturális és fizikai hatásarányok (SHAP)")

## 6. FŐRÉSZ: Integrált Eredménytár és Számítási Archívum Exportálása

### Módszertan és Funkció
A teljes ökonometriai és gépi tanulási kísérleti folyamat során a `/content/tables/` könyvtárba generált összes strukturált táblázatot (CSV, JSON, Markdown) egyetlen, letölthető ZIP archívumba csomagoljuk. Ez garantálja az empirikus levezetések és számítási eredmények teljes reprodukálhatóságát és külső AI/elemző ágensek általi feldolgozhatóságát.


In [ ]:
# ==============================================================================
# INTEGRÁLT SZÁMÍTÁSI EREDMÉNYTÁR ÉS TUDOMÁNYOS TÁBLÁZATOK ARCHIVÁLÁSA
# ==============================================================================
import shutil

# 1. Lépés: Automatikus tudományos táblázatformázás (Side-by-Side, Excel, LaTeX, HTML)
formatter_script = os.path.join(TABLES_DIR, 'build_formatted_tables.py')
if not os.path.exists(formatter_script):
    formatter_script = os.path.join(BASE_DIR, 'tables', 'build_formatted_tables.py')

if os.path.exists(formatter_script):
    try:
        log_info("Tudományos táblázatformázó motor indítása (AER/JUE szabvány)...\n")
        exec(open(formatter_script, encoding='utf-8').read())
        log_info("Táblázatformázási és Excel/LaTeX/HTML export sikeresen befejeződött.\n")
    except Exception as e:
        log_info(f"Figyelmeztetés a formázó futtatásakor: {e}\n")

# 2. Lépés: Teljes eredménytár archiválása ZIP csomagba
archive_output_base = os.path.join(BASE_DIR, "TDK_2026_Teljes_Szamitasi_Eredmenytar")
archive_zip_path = shutil.make_archive(
    base_name=archive_output_base,
    format='zip',
    root_dir=TABLES_DIR
)

file_size_kb = os.path.getsize(archive_zip_path) / 1024
log_info(f"Integrált számítási publikációs archívum sikeresen elkészült: {archive_zip_path} ({file_size_kb:.1f} KB)")
log_info("A zip tartalmazza az összes nyers CSV-t, a Side-by-Side mestertáblákat, a LaTeX .tex kódokat és az Excel munkafüzetet!")
